<sub>CELDA 01</sub>

# **HANDS - ON: DISEÑO DE AGENTES CONVERSACIONALES**

<sub>CELDA 02</sub>

Una vez vista la masterclass ***Diseño de Agentes Conversacionales***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.


De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/18wkM2LSfY39VZ2GBdcPCY2Qw0EzewJsx?usp=drive_link).

<sub>CELDA 03</sub>

## **LÓGICA DE NEGOCIO Y MODELO**

<sub>CELDA 04</sub>

### **CONFIGURACIÓN DEL ENTORNO**


<sub>CELDA 05</sub>

#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# [CELDA 06] Configuración, modos e identificadores de modelo
# Leer credenciales desde Colab Secrets — soporte de implementación, fase 1.
import sys
import os
from dataclasses import dataclass, field
from pathlib import Path

LOCAL_TEST_MODE = "LOCAL_TEST_MODE"
WHATSAPP_MODE = "WHATSAPP_MODE"
MODE = LOCAL_TEST_MODE  # No consulta Secrets de Meta ni ngrok.
IN_COLAB = "google.colab" in sys.modules
OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_MODEL = "llama3.1:8b"
STACK_URL = "http://127.0.0.1:8321"
# Stack 0.7.3 registra provider_id/provider_model_id, no el tag Ollama solo.
OLLAMA_PROVIDER_ID = "ollama"
LLAMA_STACK_MODEL = f"{OLLAMA_PROVIDER_ID}/{OLLAMA_MODEL}"  # ollama/llama3.1:8b
STACK_VERSION = "0.7.3"
CLIENT_VERSION = "0.7.3"
# Presupuesto finito por lectura HTTP de generación (no tiempo total ni de GPU).
# Colab + 8B puede ralentizar/encolar peticiones simultáneas; medir en CELDA 77.
LLM_READ_TIMEOUT_SECONDS = 600.0
RUNTIME_DIR = Path("/content/.veterinaria-phase1")

@dataclass(frozen=True)
class AppConfig:
    mode: str
    whatsapp_access_token: str | None = field(default=None, repr=False)
    whatsapp_phone_number_id: str | None = field(default=None, repr=False)
    whatsapp_verify_token: str | None = field(default=None, repr=False)
    ngrok_auth_token: str | None = field(default=None, repr=False)

    @property
    def transport_configured(self) -> bool:
        return all((self.whatsapp_access_token, self.whatsapp_phone_number_id,
                    self.whatsapp_verify_token, self.ngrok_auth_token))

def load_config(mode: str) -> AppConfig:
    if mode == LOCAL_TEST_MODE:
        return AppConfig(mode=mode)  # Sin importar userdata ni leer variables Meta.
    if mode != WHATSAPP_MODE:
        raise ValueError("Modo desconocido.")
    if not IN_COLAB:
        raise RuntimeError("La configuración WhatsApp usa Colab Secrets; ejecutar en Colab.")
    from google.colab import userdata
    # Nombres elegidos por esta implementación: el notebook no prescribe nombres.
    names = {
        "whatsapp_access_token": "WHATSAPP_ACCESS_TOKEN",
        "whatsapp_phone_number_id": "WHATSAPP_PHONE_NUMBER_ID",
        "whatsapp_verify_token": "WHATSAPP_VERIFY_TOKEN",
        "ngrok_auth_token": "NGROK_AUTH_TOKEN",
    }
    values, missing = {}, []
    for attribute, name in names.items():
        try:
            value = userdata.get(name)
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            value = None
        if not isinstance(value, str) or not value.strip():
            missing.append(name)
        else:
            values[attribute] = value.strip()
    if missing:
        raise RuntimeError("Configurar/autorizar estos Secrets: " + ", ".join(missing))
    return AppConfig(mode=mode, **values)

config = load_config(MODE)
print("CONFIG .............", config.mode)
# No mostrar config.__dict__, asdict(config) ni los valores de Secrets.

<sub>CELDA 07</sub>

#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [ ]:
# [CELDA 08] Instalación de Ollama
# Instalar Ollama — soporte de implementación, fase 1.
import shutil
import subprocess
import urllib.request

def require_colab():
    if not IN_COLAB or sys.platform != "linux":
        raise RuntimeError("Instalación preparada para Google Colab Linux; no ejecutar en Windows.")

def install_ollama():
    require_colab()
    RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
    if shutil.which("ollama"):
        print("OLLAMA_INSTALL ..... ALREADY AVAILABLE")
        return
    # El instalador oficial de Linux utiliza archivos comprimidos zstd.
    missing = [name for name in ("curl", "zstd") if not shutil.which(name)]
    if missing:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", *missing], check=True)
    installer = RUNTIME_DIR / "install-ollama.sh"
    urllib.request.urlretrieve("https://ollama.com/install.sh", installer)
    subprocess.run(["sh", str(installer)], check=True)
    if not shutil.which("ollama"):
        raise RuntimeError("Ollama no quedó disponible en PATH.")
    print("OLLAMA_INSTALL ..... OK")

install_ollama()

In [ ]:
# [CELDA 09] Arranque de Ollama y preparación del modelo
# Iniciar el servidor de Ollama en segundo plano y preparar Llama 3.1 8B.
import json
import socket
import time
from urllib.error import URLError

_processes = globals().get("_processes", {})  # Conservar al reejecutar.

def get_json(url, timeout=3):
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return json.load(response)

def service_ready(url):
    try:
        get_json(url)
        return True
    except (URLError, TimeoutError, OSError, ValueError):
        return False

def port_in_use(port):
    with socket.socket() as connection:
        connection.settimeout(0.5)
        return connection.connect_ex(("127.0.0.1", port)) == 0

def service_environment():
    return {k: v for k, v in os.environ.items()
            if not k.startswith(("WHATSAPP_", "NGROK_"))}

def start_service(name, command, ready_url, port, *, env=None, timeout=120):
    require_colab()
    if service_ready(ready_url):
        return
    process = _processes.get(name)
    if process is None or process.poll() is not None:
        if port_in_use(port):
            raise RuntimeError(f"Puerto {port} ocupado por un servicio no listo; revisar.")
        RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
        with (RUNTIME_DIR / f"{name}.log").open("ab") as log:
            process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT,
                                       env=env, start_new_session=True)
        _processes[name] = process
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError(f"{name} terminó; revisar su log en {RUNTIME_DIR}. No publicarlo sin revisar.")
        if service_ready(ready_url):
            return
        time.sleep(0.5)  # Polling acotado; no espera fija de arranque.
    raise TimeoutError(f"{name} no estuvo listo en {timeout}s; reejecutar reutiliza el proceso.")

def prepare_ollama_model():
    require_colab()
    environment = service_environment()
    environment["OLLAMA_HOST"] = "127.0.0.1:11434"
    start_service("ollama", ["ollama", "serve"], OLLAMA_URL + "/api/tags", 11434, env=environment)
    available = {m.get("name") for m in get_json(OLLAMA_URL + "/api/tags").get("models", [])}
    if OLLAMA_MODEL not in available:
        print("LLAMA_MODEL ........ DOWNLOADING (puede tardar varios minutos)")
        subprocess.run(["ollama", "pull", OLLAMA_MODEL], env=environment, check=True)
    available = {m.get("name") for m in get_json(OLLAMA_URL + "/api/tags").get("models", [])}
    if OLLAMA_MODEL not in available:
        raise RuntimeError("El modelo requerido no está disponible en Ollama.")
    print("OLLAMA ............. OK")
    print("LLAMA_MODEL ........ OK:", OLLAMA_MODEL)

prepare_ollama_model()

<sub>CELDA 10</sub>

#### **INSTALAR Y LEVANTAR LLAMA STACK**

La versión de Python no garantiza que `ensurepip` esté disponible. En la celda
siguiente se comprueban intérprete, módulos y estado real del venv. Se crea con
`venv --without-pip` y se instala pip **dentro** del venv mediante `pip --python`,
procedimiento soportado que no necesita instalar `python3-venv` por suposición ni
cambiar los paquetes del kernel. Pip de bootstrap: 25.3, compatible con Python 3.13.
Se requiere que el pip del kernel admita `--python` (>=22.3); se verifica su ayuda.
Si falta el propio módulo `venv`, se muestra el error y se detiene: no se adivina
qué paquete del sistema corresponde al intérprete activo.

Un venv válido se reutiliza. Si solo falta pip, se repara en el mismo directorio.
Si está incompleto o usa otro intérprete/aislamiento, se conserva como
`stack-env.invalid-<id>` y se crea de nuevo. No se borran bases de datos ni otros
archivos del runtime. No reparar con el servidor Stack activo.

El error original (exit status 1 sin stdout/stderr) no permite confirmar que
`ensurepip` fuera la causa; su ausencia es una hipótesis, no un diagnóstico probado.
Esta preparación evita esa dependencia y muestra stdout/stderr si vuelve a fallar.

Referencias: [venv de Python](https://docs.python.org/3.13/library/venv.html),
[pip --python](https://pip.pypa.io/en/stable/topics/python-option/).


In [ ]:
# [CELDA 11] Preparación del venv e instalación de dependencias
# Instalar dependencias — servidor aislado para no reemplazar paquetes de Colab.
from importlib import metadata
from uuid import uuid4

STACK_ENV = RUNTIME_DIR / "stack-env"
STACK_PYTHON = STACK_ENV / "bin/python"

STACK_BOOTSTRAP_PIP = "25.3"  # Soporta Python 3.13; no cambia pip del kernel.


def run_stack_setup(command, *, check=True):
    # Colab muestra ambos streams incluso si subprocess termina con error.
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end="", flush=True)
    if result.stderr:
        print(result.stderr, end="", file=sys.stderr, flush=True)
    if check:
        result.check_returncode()
    return result


def stack_venv_usable(*, with_pip):
    if not STACK_PYTHON.is_file() or not (STACK_ENV / "pyvenv.cfg").is_file():
        return False
    # No basta bin/python: exigir prefijo aislado, versión/base correctas y pip local.
    probe = """
import sys, site
from pathlib import Path
expected = Path(sys.argv[1]).resolve()
assert Path(sys.prefix).resolve() == expected, 'Prefijo de venv incorrecto'
assert sys.prefix != sys.base_prefix, 'No es un entorno virtual'
assert list(sys.version_info[:2]) == [int(v) for v in sys.argv[2].split('.')], 'Version distinta'
assert Path(sys._base_executable).resolve() == Path(sys.argv[3]).resolve(), 'Interprete base distinto'
assert not site.ENABLE_USER_SITE, 'User site habilitado'
cfg = (expected / 'pyvenv.cfg').read_text().lower()
assert 'include-system-site-packages = false' in cfg, 'Paquetes del sistema habilitados'
if sys.argv[4] == 'pip':
    import pip
    assert Path(pip.__file__).resolve().is_relative_to(expected), 'pip fuera del venv'
"""
    try:
        result = run_stack_setup([
            str(STACK_PYTHON), "-I", "-c", probe, str(STACK_ENV),
            f"{sys.version_info.major}.{sys.version_info.minor}",
            str(Path(sys._base_executable).resolve()), "pip" if with_pip else "base",
        ], check=False)
        if result.returncode:
            return False
        if with_pip:
            return run_stack_setup([str(STACK_PYTHON), "-I", "-m", "pip", "--version"],
                                   check=False).returncode == 0
        return True
    except OSError as error:
        print(f"VENV_PROBE ......... {type(error).__name__}: {error}")
        return False


def prepare_stack_venv():
    # Solo se prepara/repara el directorio reservado, nunca todo RUNTIME_DIR.
    root = RUNTIME_DIR.resolve()
    expected = root / "stack-env"
    if STACK_ENV.is_symlink() or STACK_ENV.resolve() != expected:
        raise RuntimeError("Ruta de venv inesperada o enlace: no se modificará.")
    run_stack_setup([sys.executable, "-I", "-c",
        "import sys,importlib.util; "
        "print('VENV_RUNTIME:',sys.executable,sys.version.split()[0]); "
        "print('VENV_MODULE:',importlib.util.find_spec('venv') is not None); "
        "print('ENSUREPIP_MODULE:',importlib.util.find_spec('ensurepip') is not None)"])
    if stack_venv_usable(with_pip=True):
        print("STACK_VENV ........ REUSED (aislamiento y pip verificados)")
        return
    if port_in_use(8321):
        raise RuntimeError("Detener Llama Stack antes de reparar su entorno virtual.")
    # pip >=22.3 soporta --python incluso si el venv todavía no tiene pip.
    # Comprobar capacidad real antes de modificar el entorno. No instalar apt a ciegas.
    help_result = subprocess.run([sys.executable, "-I", "-m", "pip", "--help"],
                                 text=True, capture_output=True)
    if help_result.returncode or "--python" not in help_result.stdout:
        print(help_result.stdout, end="")
        print(help_result.stderr, end="", file=sys.stderr)
        raise RuntimeError("Se requiere pip del kernel con --python (>=22.3) para bootstrap aislado.")
    if not stack_venv_usable(with_pip=False):
        root.mkdir(parents=True, exist_ok=True)
        if STACK_ENV.exists():
            # Conservar el entorno inválido para diagnóstico; no borrar recursivamente.
            backup = root / f"stack-env.invalid-{uuid4().hex}"
            STACK_ENV.rename(backup)
            print("STACK_VENV_BACKUP ..", backup)
        # Evitar dependencia de ensurepip/wheels del sistema, conservando venv nativo.
        run_stack_setup([sys.executable, "-I", "-m", "venv", "--without-pip", str(STACK_ENV)])
        if not stack_venv_usable(with_pip=False):
            raise RuntimeError("venv no creó un entorno aislado utilizable; revisar salida anterior.")
    # Instalar exclusivamente en el intérprete del venv. No usar --system-site-packages.
    run_stack_setup([sys.executable, "-I", "-m", "pip", "--isolated", "--python", str(STACK_PYTHON),
                     "install", "--disable-pip-version-check", "--force-reinstall",
                     f"pip=={STACK_BOOTSTRAP_PIP}"])
    if not stack_venv_usable(with_pip=True):
        raise RuntimeError("Bootstrap de pip incompleto; revisar stdout/stderr anteriores.")
    print("STACK_VENV ........ READY (aislamiento y pip verificados)")


def install_stack_dependencies():
    require_colab()
    if sys.version_info < (3, 12):
        raise RuntimeError("Llama Stack 0.7.3 requiere Python >=3.12. Usar un runtime Colab compatible.")
    prepare_stack_venv()
    # Servidor/API/cliente alineados; solo dependencias de los providers utilizados.
    subprocess.run([str(STACK_PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
                    f"llama-stack=={STACK_VERSION}", f"llama-stack-api=={STACK_VERSION}",
                    f"llama-stack-client=={CLIENT_VERSION}", "ollama==0.6.3",
                    "matplotlib", "pillow>=12.2.0", "scikit-learn", "mcp>=1.23.0,<2"], check=True)
    imported = sys.modules.get("llama_stack_client")
    if imported is not None and getattr(imported, "__version__", None) != CLIENT_VERSION:
        raise RuntimeError("Otra SDK ya importada: reiniciar el runtime antes de instalar el cliente.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                    f"llama-stack-client=={CLIENT_VERSION}"], check=True)
    if metadata.version("llama-stack-client") != CLIENT_VERSION:
        raise RuntimeError("Versión inesperada del cliente.")
    print("STACK_PACKAGES ..... INSTALLED; readiness pendiente de la siguiente celda")

install_stack_dependencies()

In [ ]:
# [CELDA 12] Preparación del cliente Llama Stack
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado.
# Dependencias instaladas en la celda anterior. Sin generar conversaciones.
import inspect

def build_stack_config():
    # Persistencia nativa de Stack: el agente NO consulta SQLite ni crea memoria
    # alternativa. Las conversaciones se administrarán exclusivamente por su API.
    return {
        "version": 2, "distro_name": "veterinaria-colab",
        "apis": ["inference", "responses", "vector_io", "tool_runtime", "files"],
        "providers": {
            "inference": [{"provider_id": OLLAMA_PROVIDER_ID, "provider_type": "remote::ollama",
                           "config": {"base_url": OLLAMA_URL + "/v1"}}],
            "responses": [{"provider_id": "builtin", "provider_type": "inline::builtin",
                           "config": {"persistence": {
                               "agent_state": {"backend": "kv_default", "namespace": "agents"},
                               "responses": {"backend": "sql_default", "table_name": "responses"}}}}],
            "files": [{"provider_id": "localfs", "provider_type": "inline::localfs",
                       "config": {"storage_dir": str(RUNTIME_DIR / "files"),
                                  "metadata_store": {"backend": "sql_default", "table_name": "files"}}}],
            # Dependencias internas de responses, sin RAG ni herramientas.
            "vector_io": [], "tool_runtime": [],
        },
        "storage": {"backends": {
            "kv_default": {"type": "kv_sqlite", "db_path": str(RUNTIME_DIR / "stack-kv.db")},
            "sql_default": {"type": "sql_sqlite", "db_path": str(RUNTIME_DIR / "stack-sql.db")},
        }, "stores": {
            "metadata": {"backend": "kv_default", "namespace": "registry"},
            "inference": {"backend": "sql_default", "table_name": "inference"},
            "conversations": {"backend": "sql_default", "table_name": "conversations"},
            "responses": {"backend": "sql_default", "table_name": "responses"},
            "prompts": {"backend": "kv_default", "namespace": "prompts"},
            "connectors": {"backend": "kv_default", "namespace": "connectors"},
        }},
        "registered_resources": {"models": [{"model_id": LLAMA_STACK_MODEL, "provider_id": OLLAMA_PROVIDER_ID,
                                             "provider_model_id": OLLAMA_MODEL, "model_type": "llm"}]},
        "server": {"host": "127.0.0.1", "port": 8321, "workers": 1},
    }


class ModelRegistryError(RuntimeError):
    """Diagnóstico seguro: solo forma de respuesta e identificadores públicos."""


def require_registered_stack_model(models):
    # SDK 0.7.3: ModelListResponse = List[Model], Model.id: str.
    # models.list aplica DataWrapper._unwrapper; NO devuelve una página.
    response_type = type(models).__name__
    if not isinstance(models, list):
        raise ModelRegistryError(
            f"models.list(): tipo={response_type}; cantidad=desconocida; "
            "ids=no disponibles; se esperaba list[Model] en SDK 0.7.3.")
    model_ids = [getattr(model, "id", None) for model in models]
    valid_ids = [model_id for model_id in model_ids if isinstance(model_id, str) and model_id.strip()]
    diagnostic = f"tipo={response_type}; cantidad={len(models)}; ids={valid_ids!r}"
    if len(valid_ids) != len(models):
        raise ModelRegistryError(f"Registro con elementos sin id string válido; {diagnostic}")
    if LLAMA_STACK_MODEL not in model_ids:
        raise ModelRegistryError(f"El modelo requerido {LLAMA_STACK_MODEL!r} no está registrado; {diagnostic}")


def stack_config_matches_running_contract(actual, expected):
    if actual == expected:
        return True
    # Única migración equivalente admitida: el antiguo model_id sin prefijo.
    # Stack 0.7.3 construye el ID desde provider_id/provider_model_id en ambos casos.
    legacy = json.loads(json.dumps(expected))
    legacy["registered_resources"]["models"][0]["model_id"] = OLLAMA_MODEL
    return actual == legacy


def prepare_stack_client():
    require_colab()
    if not service_ready(OLLAMA_URL + "/api/tags"):
        raise RuntimeError("Preparar Ollama y el modelo antes de Llama Stack.")
    stack_config = RUNTIME_DIR / "stack-config.json"
    serialized = json.dumps(build_stack_config(), indent=2)
    if stack_config.exists() and port_in_use(8321):
        actual_config = json.loads(stack_config.read_text())
        if not stack_config_matches_running_contract(actual_config, build_stack_config()):
            raise RuntimeError("La configuración cambió con Stack activo. Reiniciar el servicio.")
        if actual_config != build_stack_config():
            print("STACK_CONFIG ....... ID explícito actualizado; mismo modelo/provider activo")
    stack_config.write_text(serialized)
    # Esquema REAL instalado. JSON también es YAML válido para la CLI.
    validate = ("import json,sys; from llama_stack.core.datatypes import StackConfig; "
                "StackConfig.model_validate(json.load(open(sys.argv[1])))")
    subprocess.run([str(STACK_PYTHON), "-c", validate, str(stack_config)], check=True)
    start_service("llama-stack", [str(STACK_ENV / "bin/llama"), "stack", "run", str(stack_config)],
                  STACK_URL + "/v1/health", 8321, env=service_environment(), timeout=180)
    if get_json(STACK_URL + "/v1/version").get("version") != STACK_VERSION:
        raise RuntimeError("El servidor activo no coincide con la versión fijada.")
    if metadata.version("llama-stack-client") != CLIENT_VERSION:
        raise RuntimeError("Cliente incompatible con la versión fijada.")
    # Crear cliente SOLO después de readiness.
    from llama_stack_client import LlamaStackClient
    candidate = LlamaStackClient(base_url=STACK_URL, timeout=120.0, max_retries=0)
    try:
        required = [candidate.conversations.create, candidate.conversations.items.list,
                    candidate.conversations.items.delete, candidate.responses.create]
        if not all(callable(method) for method in required):
            raise RuntimeError("La SDK instalada no expone las APIs requeridas.")
        inspect.signature(candidate.conversations.create).bind()
        if "conversation" not in inspect.signature(candidate.responses.create).parameters:
            raise RuntimeError("responses.create no admite conversation.")
        paths = get_json(STACK_URL + "/openapi.json").get("paths", {})
        if any("post" not in paths.get(route, {}) for route in ("/v1/conversations", "/v1/responses")):
            raise RuntimeError("El servidor no expone conversations/responses.")
        require_registered_stack_model(candidate.models.list())
    except Exception:
        candidate.close()
        raise
    previous = globals().get("client")
    if previous is not None:
        previous.close()
    print("LLAMA_STACK ........ OK")
    print("CONVERSATIONS_API .. AVAILABLE (sin crear conversaciones)")
    return candidate

client = prepare_stack_client()

<sub>CELDA 13</sub>

### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [ ]:
# [CELDA 14] Configuración compartida del transporte WhatsApp
# Ejecutar/redefinir transporte solo con servidor detenido. El núcleo no cambia.
import importlib.util
import subprocess
import sys
if globals().get('_wa_thread') is not None and _wa_thread.is_alive():
    raise RuntimeError("Detener primero en CELDA 34 o 65; no redefinir un servidor activo.")
_transport_packages = {'fastapi': 'fastapi>=0.115,<1', 'uvicorn': 'uvicorn>=0.30,<1',
                       'httpx': 'httpx>=0.27,<1', 'pyngrok': 'pyngrok>=7,<9'}
_missing = [package for module, package in _transport_packages.items()
            if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *_missing], check=True)

from fastapi import FastAPI, BackgroundTasks, Request, HTTPException
from fastapi.responses import PlainTextResponse
import httpx
import json
import re
import hmac
import hashlib
import secrets
import threading
import time
import logging
from collections import deque, OrderedDict
from uuid import uuid4
from dataclasses import dataclass, field

WHATSAPP_GRAPH_VERSION = 'v23.0'  # Versión explícita de Graph, independiente de Stack.
WHATSAPP_HTTP_TIMEOUT = httpx.Timeout(30.0, connect=10.0, pool=10.0)
WEBHOOK_PORT = 8000
MAX_WEBHOOK_BYTES = 1_000_000
# Evitar query strings (verify token), headers y cuerpos en logs de bibliotecas.
for _logger in ('httpx', 'httpcore', 'pyngrok'):
    logging.getLogger(_logger).disabled = True

_wa_events = deque(maxlen=500)
_wa_guard = threading.RLock()
_wa_key = secrets.token_bytes(32)
_wa_seen = OrderedDict()  # IDs deduplicados solo en esta sesión; no memoria del agente.
_wa_pending = set()
_wa_server = _wa_thread = _wa_tunnel = _wa_ngrok_config = None
_wa_previous_diagnostics = None
_wa_app_secret = None

class TransportError(RuntimeError):
    pass

@dataclass(frozen=True)
class IncomingText:
    phone: str = field(repr=False)
    text: str = field(repr=False)
    message_id: str | None = field(default=None, repr=False)

def transport_ref(value):
    return hmac.new(_wa_key, value.encode(), hashlib.sha256).hexdigest()[:20]

def transport_event(event, **fields):
    allowed = {'job', 'message_ref', 'phone_ref', 'run_id', 'route', 'history_count',
               'history_trimmed', 'conversation_ref', 'llm_requests', 'error_type', 'count'}
    if not fields.keys() <= allowed:
        raise ValueError('Campos de diagnóstico no permitidos.')
    with _wa_guard:
        _wa_events.append({'event': event, 'timestamp_ns': time.monotonic_ns(), **fields})

def transport_snapshot():
    with _wa_guard:
        return [dict(row) for row in _wa_events]

def require_whatsapp_mode():
    # Ambas guardas: cambiar únicamente MODE no habilita un config local.
    if MODE != WHATSAPP_MODE or config.mode != WHATSAPP_MODE:
        raise TransportError('Transporte bloqueado fuera de WHATSAPP_MODE.')
    if not config.transport_configured:
        raise TransportError('Faltan Secrets del transporte.')

app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
print('TRANSPORT: IMPLEMENTED — PENDING META VALIDATION; servidor aún detenido.')


In [ ]:
# [CELDA 15] normalize_phone()
# Definir función central de normalización — compartida por ambos modos.
import re

def normalize_phone(phone_number: str) -> str:
    '''Clave internacional +dígitos; entrada sin '+' debe incluir código de país.

    Validación sintáctica, no certifica asignación ni existencia del número.
    No añade país ni aplica conversiones exclusivas de México.
    '''
    if not isinstance(phone_number, str):
        raise TypeError("El teléfono debe ser string.")
    value = phone_number.strip()
    if not value or not re.fullmatch(r"[+0-9().\s-]+", value):
        raise ValueError("Teléfono vacío o con caracteres no permitidos.")
    value = re.sub(r"[().\s-]", "", value)
    if value.startswith("00"):
        value = "+" + value[2:]
    digits = value[1:] if value.startswith("+") else value
    if not re.fullmatch(r"[1-9][0-9]{6,14}", digits):
        raise ValueError("Usar entre 7 y 15 dígitos con código internacional, sin extensiones.")
    return "+" + digits

<sub>CELDA 16</sub>

#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Cada teléfono normalizado localiza una conversación real de Llama Stack. El índice
local solo guarda su ID; los mensajes y el contexto residen en Stack. Un lock por
teléfono protege la creación y los turnos completos. Ejecutar las celdas en orden.

In [ ]:
# [CELDA 17] Contratos, diagnóstico, conversaciones y locks por teléfono
# Contratos únicos, diagnóstico y estado por teléfono — fases 1–3.
from threading import Lock, RLock
import hashlib
import hmac
import secrets

# Solo estos números ficticios pueden aparecer sin anonimizar en diagnóstico.
SYNTHETIC_PHONES = frozenset(f"+1555000000{i}" for i in range(1, 10))
_DIAGNOSTIC_KEY = secrets.token_bytes(32)
from collections import deque
from dataclasses import dataclass, field
from enum import Enum
from uuid import uuid4
import time

class Route(str, Enum):
    LLM = "LLM"
    HUMAN_ESCALATION = "HUMAN_ESCALATION"
    VETERINARY_EMERGENCY = "VETERINARY_EMERGENCY"

@dataclass(frozen=True)
class CoreResult:
    response_text: str
    normalized_phone: str = field(repr=False)
    conversation_id: str | None = field(repr=False)
    route: Route
    history_count: int | None  # None = no observado, no un cero inventado.
    run_id: str
    history_trimmed: bool | None = None  # None: la ruta no inspeccionó historial.

class EventKind(str, Enum):
    RUN_STARTED = "RUN_STARTED"
    LOCK_WAIT = "LOCK_WAIT"
    LOCK_ACQUIRED = "LOCK_ACQUIRED"
    LOCK_RELEASED = "LOCK_RELEASED"
    LLM_REQUEST_STARTED = "LLM_REQUEST_STARTED"
    LLM_REQUEST_COMPLETED = "LLM_REQUEST_COMPLETED"
    LLM_REQUEST_FAILED = "LLM_REQUEST_FAILED"
    LLM_SKIPPED = "LLM_SKIPPED"
    RUN_FAILED = "RUN_FAILED"
    HISTORY_UNAVAILABLE = "HISTORY_UNAVAILABLE"
    HISTORY_OBSERVED = "HISTORY_OBSERVED"
    HISTORY_TRIM_STARTED = "HISTORY_TRIM_STARTED"
    HISTORY_TRIM_COMPLETED = "HISTORY_TRIM_COMPLETED"
    HISTORY_TRIM_FAILED = "HISTORY_TRIM_FAILED"
    CONVERSATION_ROTATED = "CONVERSATION_ROTATED"
    ROUTE_SELECTED = "ROUTE_SELECTED"
    RUN_FINISHED = "RUN_FINISHED"

@dataclass(frozen=True)
class DiagnosticEvent:
    event: EventKind
    run_id: str
    timestamp_ns: int
    normalized_phone: str | None = field(default=None, repr=False)
    conversation_id: str | None = field(default=None, repr=False)
    route: Route | None = None
    history_count: int | None = None
    lock_id: str | None = None
    phone_ref: str | None = None
    sequence: int = 0
    history_before: int | None = None
    history_after: int | None = None
    old_conversation_id: str | None = field(default=None, repr=False)
    new_conversation_id: str | None = field(default=None, repr=False)

class DiagnosticRecorder:
    # Registro acotado y opt-in, NO memoria conversacional.
    # En pruebas usar solo teléfonos sintéticos. No registrar texto/payloads/Secrets.
    def __init__(self, *, enabled=False, capacity=1000):
        self.enabled = enabled
        self._events = deque(maxlen=capacity)
        self._guard = RLock()
        self._sequence = 0

    def record(self, event: EventKind, *, run_id: str, normalized_phone=None,
               conversation_id=None, route=None, history_count=None, lock_id=None,
               history_before=None, history_after=None,
               old_conversation_id=None, new_conversation_id=None):
        if not self.enabled:
            return
        if not isinstance(event, EventKind):
            raise TypeError("Evento no permitido.")
        if route is not None and not isinstance(route, Route):
            raise TypeError("Ruta no permitida.")
        if history_count is not None and (type(history_count) is not int or history_count < 0):
            raise ValueError("history_count debe ser entero no negativo o None.")
        phone_ref = (hmac.new(_DIAGNOSTIC_KEY, normalized_phone.encode(), hashlib.sha256).hexdigest()[:20]
                     if normalized_phone else None)
        safe_phone = normalized_phone if normalized_phone in SYNTHETIC_PHONES else None
        with self._guard:
            self._sequence += 1
            self._events.append(DiagnosticEvent(
                event, run_id, time.monotonic_ns(), safe_phone,
                conversation_id, route, history_count, lock_id, phone_ref,
                self._sequence, history_before, history_after,
                old_conversation_id, new_conversation_id))

    def release_phone_lock(self, phone_lock, **fields):
        # Liberación real + evento ordenados frente a otros record().
        # Solo retiene el guard durante release/registro, nunca durante generación.
        with self._guard:
            phone_lock.release()
            self.record(EventKind.LOCK_RELEASED, **fields)

    def snapshot(self):
        with self._guard:
            return tuple(self._events)

    def clear(self, *, phones):
        # Reset selectivo, nunca vaciar los diagnósticos de otros usuarios.
        with self._guard:
            kept = [e for e in self._events if e.normalized_phone not in phones]
            self._events.clear()
            self._events.extend(kept)


def new_run_id() -> str:
    return uuid4().hex

diagnostics = DiagnosticRecorder(enabled=False)


# Índices locales; NO guardan mensajes ni reconstruyen contexto.
# Reejecutar esta celda reinicia índices: hacerlo solo con el núcleo inactivo.
_conversation_ids = {}
_phone_locks = {}
_lock_registry_guard = Lock()


def get_phone_lock(normalized_phone):
    with _lock_registry_guard:
        if normalized_phone not in _phone_locks:
            _phone_locks[normalized_phone] = (Lock(), uuid4().hex)
        return _phone_locks[normalized_phone]


def get_or_create_conversation(normalized_phone):
    # El llamador debe poseer el lock de este teléfono.
    conversation_id = _conversation_ids.get(normalized_phone)
    if conversation_id is None:
        conversation = client.conversations.create()
        conversation_id = conversation.id
        if not isinstance(conversation_id, str) or not conversation_id.strip():
            raise RuntimeError("Stack no devolvió un ID de conversación válido.")
        _conversation_ids[normalized_phone] = conversation_id
    return conversation_id

<sub>CELDA 18</sub>

#### **GENERAR LA RESPUESTA CON HISTORIAL Y UNA REGLA FIJA**

La petición explícita de atención humana se resuelve antes del modelo. En los demás
casos se envía únicamente el mensaje actual con `conversation=ID`: Stack recupera
la memoria. No hay lista Python de mensajes ni reenvío manual del historial.

In [ ]:
# [CELDA 19] Reglas deterministas, generación y recorte de historial
# Detector determinista y generación con la SDK real 0.7.3.
import unicodedata
import httpx

HUMAN_ESCALATION_RESPONSE = (
    "Tu solicitud es de atención humana del equipo de la veterinaria. "
    "Para recibirla, contacta directamente a la clínica; este chat local no envía avisos al equipo."
)
VETERINARY_INSTRUCTIONS = (
    "Eres el asistente de una veterinaria. Responde en español, de forma breve y clara, "
    "sobre servicios veterinarios, cuidado general de mascotas y consultas administrativas. "
    "Usa el contexto de la conversación. No inventes horarios, precios, citas ni diagnósticos "
    "clínicos definitivos; pide los datos que falten y recomienda valoración profesional cuando corresponda."
)


def normalized_text(text):
    decomposed = unicodedata.normalize("NFKD", text.casefold())
    return " ".join("".join(c for c in decomposed if not unicodedata.combining(c)).split())


def requests_human(message):
    # Intenciones completas; no basta con mencionar 'persona' o 'humano'.
    # fullmatch evita citas, negaciones y menciones incidentales dentro de otra frase.
    text = normalized_text(message).strip(" .!¡?¿,;")
    return re.fullmatch(
        r"(?:por favor[, ]+)?(?:"
        r"(?:quiero|necesito|quisiera) hablar con (?:una persona|un humano|alguien)|"
        r"(?:comunicame|comuniqueme|ponme en contacto) con (?:una persona|un humano|alguien)|"
        r"necesito atencion humana"
        r")(?:[, ]+por favor)?", text
    ) is not None


# Disparadores explícitos y acotados; no es un clasificador clínico.
# Se busca presencia con límites de palabra. No interpreta negación, citas ni
# temporalidad: una mención histórica puede activar la regla conservadora.
EMERGENCY_PATTERNS = (
    r"\bno respira\b", r"\bdificultad para respirar\b", r"\bno puede respirar\b",
    r"\bconvulsi(?:on|ones|onando)\b", r"\bsangrado abundante\b", r"\bhemorragia(?:s)?\b",
    r"\batropellad[oa]s?\b", r"\batropellaron\b", r"\benvenenamiento(?:s)?\b",
    r"\benvenenad[oa]s?\b", r"\bintoxicacion grave\b", r"\binconsciente(?:s)?\b",
    r"\bperdio el conocimiento\b",
)
VETERINARY_EMERGENCY_RESPONSE = (
    "Puede tratarse de una urgencia. Busca atención veterinaria inmediata. "
    "Este chat no sustituye la atención veterinaria presencial."
)


def is_veterinary_emergency(message):
    text = normalized_text(message)
    return any(re.search(pattern, text) is not None for pattern in EMERGENCY_PATTERNS)


def select_route(message):
    # DECISIÓN DE IMPLEMENTACIÓN, no prioridad textual del notebook original.
    if is_veterinary_emergency(message):
        return Route.VETERINARY_EMERGENCY
    if requests_human(message):
        return Route.HUMAN_ESCALATION
    return Route.LLM


def extract_response_text(response):
    # ResponseObject.output -> message.content: str o bloques output_text.
    # No recorrer herramientas, reasoning, inputs ni serializar objetos completos.
    if getattr(response, "status", None) != "completed" or getattr(response, "error", None) is not None:
        raise RuntimeError("La generación de Stack no terminó correctamente.")
    output = getattr(response, "output", None)
    if not isinstance(output, list) or len(output) > 100:
        raise RuntimeError("Estructura de salida de Stack no utilizable.")
    parts = []
    for item in output:
        if getattr(item, "type", None) != "message" or getattr(item, "role", None) != "assistant":
            continue
        content = getattr(item, "content", None)
        if isinstance(content, str):
            parts.append(content)
        elif isinstance(content, list) and len(content) <= 100:
            for block in content:
                if getattr(block, "type", None) == "output_text":
                    text = getattr(block, "text", None)
                    if isinstance(text, str):
                        parts.append(text)
        else:
            raise RuntimeError("Contenido de respuesta no utilizable.")
    if sum(map(len, parts)) > 32000:
        raise RuntimeError("Texto de respuesta excede el tamaño permitido.")
    text = "\n".join(p.strip() for p in parts if p.strip())
    if not text:
        raise RuntimeError("Stack no devolvió texto final utilizable.")
    return text


def generate_response(message, conversation_id, emit):
    # Sobrescritura por request: no mutar el cliente compartido ni sus otros endpoints.
    timeout = httpx.Timeout(120.0, read=LLM_READ_TIMEOUT_SECONDS)
    emit(EventKind.LLM_REQUEST_STARTED)
    try:
        response = client.responses.create(
            model=LLAMA_STACK_MODEL, input=message, conversation=conversation_id,
            instructions=VETERINARY_INSTRUCTIONS, stream=False, store=True,
            temperature=0, max_output_tokens=384, timeout=timeout,
        )
        text = extract_response_text(response)
    except Exception:
        # No registrar excepciones completas: pueden contener mensajes/payloads.
        emit(EventKind.LLM_REQUEST_FAILED)
        raise
    emit(EventKind.LLM_REQUEST_COMPLETED)
    return text


# DECISIÓN DE IMPLEMENTACIÓN: seis items API, no seis turnos.
MAX_HISTORY_ITEMS = 6
HISTORY_INSPECTION_LIMIT = 100


def read_history_items(conversation_id):
    # En el servidor 0.7.3 list_items ignora after y devuelve has_more=False.
    # No asumir que una página llena representa todo el historial.
    page = client.conversations.items.list(
        conversation_id, limit=HISTORY_INSPECTION_LIMIT, order="asc")
    items = page.data
    if (not isinstance(items, list) or len(items) >= HISTORY_INSPECTION_LIMIT
            or page.has_more is not False):
        raise RuntimeError("Historial no verificable: página llena o incompleta en Stack 0.7.3.")
    ids = [getattr(item, "id", None) for item in items]
    if any(not isinstance(item_id, str) or not item_id for item_id in ids) or len(set(ids)) != len(ids):
        raise RuntimeError("Items de historial sin IDs únicos válidos.")
    return items


def history_item_payload(item):
    # Transferencia transitoria de texto vía API, nunca memoria Python persistente.
    # No reutilizar IDs: add_items de Stack hace upsert por ID global.
    role, content = getattr(item, "role", None), getattr(item, "content", None)
    if getattr(item, "type", None) != "message" or role not in ("user", "assistant"):
        raise RuntimeError("La rotación de este agente admite únicamente mensajes de texto.")
    if isinstance(content, str):
        copied = content
    elif isinstance(content, list):
        copied = []
        for block in content:
            kind, text = getattr(block, "type", None), getattr(block, "text", None)
            if kind not in ("input_text", "output_text") or not isinstance(text, str):
                raise RuntimeError("Item no textual: no se puede transferir sin perder contexto.")
            copied.append({"type": kind, "text": text})
    else:
        raise RuntimeError("Contenido no compatible con la transferencia.")
    return {"type": "message", "role": role, "content": copied}


def enforce_history_limit(phone, conversation_id, emit):
    # Siempre dentro del lock del teléfono. Ningún éxito si no podemos verificar.
    try:
        items = read_history_items(conversation_id)
    except Exception:
        emit(EventKind.HISTORY_UNAVAILABLE)
        raise
    before = len(items)
    emit(EventKind.HISTORY_OBSERVED, history_count=before)
    if before <= MAX_HISTORY_ITEMS:
        return conversation_id, before, False
    new_id = None
    emit(EventKind.HISTORY_TRIM_STARTED, history_before=before,
         old_conversation_id=conversation_id)
    try:
        retained = items[-MAX_HISTORY_ITEMS:]
        payloads = [history_item_payload(item) for item in retained]
        # Borrar items NO invalida conversation_messages del provider responses
        # 0.7.3. Se necesita un ID nuevo para que la próxima generación reconstruya
        # su contexto exclusivamente desde estos seis items nativos.
        candidate = client.conversations.create(items=payloads)
        new_id = candidate.id
        if not isinstance(new_id, str) or not new_id or new_id == conversation_id:
            raise RuntimeError("Stack no devolvió un ID nuevo válido para rotación.")
        copied = read_history_items(new_id)
        if [history_item_payload(item) for item in copied] != payloads:
            raise RuntimeError("Stack no conservó exactamente los items transferidos y su orden.")
        if {item.id for item in copied} & {item.id for item in items}:
            raise RuntimeError("Stack reutilizó IDs de items al transferir contexto.")
        # Publicar solo una conversación completa/verificada. Si falla el borrado
        # posterior, el run falla, pero el índice ya apunta al contexto limitado.
        _conversation_ids[phone] = new_id
        emit(EventKind.CONVERSATION_ROTATED, history_before=before,
             history_after=len(copied), old_conversation_id=conversation_id,
             new_conversation_id=new_id)
        # Preferencia de borrado real: eliminar los items antiguos del ID anterior.
        # NO afirmar borrado de sus responses/caché ni de la conversación abandonada.
        removed_ids = {item.id for item in items[:-MAX_HISTORY_ITEMS]}
        for item in items[:-MAX_HISTORY_ITEMS]:
            deleted = client.conversations.items.delete(item.id, conversation_id=conversation_id)
            if deleted.id != item.id or deleted.deleted is False:
                raise RuntimeError("Stack no confirmó la eliminación del item esperado.")
        old_remaining = read_history_items(conversation_id)
        if ({item.id for item in old_remaining} != {item.id for item in retained}
                or removed_ids & {item.id for item in old_remaining}):
            raise RuntimeError("El borrado real de items no quedó verificado.")
        final = read_history_items(new_id)
        if [history_item_payload(item) for item in final] != payloads:
            raise RuntimeError("La conversación activa cambió durante el recorte.")
        emit(EventKind.HISTORY_TRIM_COMPLETED, history_before=before,
             history_after=len(final), history_count=len(final),
             old_conversation_id=conversation_id, new_conversation_id=new_id)
        return new_id, len(final), True
    except Exception:
        emit(EventKind.HISTORY_TRIM_FAILED, history_before=before,
             old_conversation_id=conversation_id, new_conversation_id=new_id)
        raise

<sub>CELDA 20</sub>

## **DESPLIEGUE DEL SERVIDOR**

<sub>CELDA 21</sub>

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — reduciendo reintentos por demora. Esto no garantiza entrega exactamente una vez; el adaptador deduplica IDs de forma acotada en la sesión.

In [ ]:
# [CELDA 22] Webhook GET: verificación segura
@app.get('/webhook')
def verify_webhook(request: Request):
    try:
        require_whatsapp_mode()
    except TransportError:
        raise HTTPException(503, 'Transporte deshabilitado.') from None
    query = request.query_params
    supplied = query.get('hub.verify_token', '')
    if (query.get('hub.mode') != 'subscribe'
            or not hmac.compare_digest(supplied.encode(), config.whatsapp_verify_token.encode())
            or not query.get('hub.challenge')):
        transport_event('VERIFY_REJECTED')
        raise HTTPException(403, 'Verificación rechazada.')
    transport_event('VERIFY_ACCEPTED')
    return PlainTextResponse(query['hub.challenge'])


In [ ]:
# [CELDA 23] Webhook POST: extracción y delegación
def extract_text_messages(payload, expected_phone_id):
    # Envelope real: entry[].changes[].value.messages[].text.body.
    # Ignora estados, medios, estructuras incompletas y otro número receptor.
    def objects(value):
        return [v for v in value if isinstance(v, dict)] if isinstance(value, list) else []
    if not isinstance(payload, dict) or payload.get('object') != 'whatsapp_business_account':
        return []
    found = []
    for entry in objects(payload.get('entry')):
        for change in objects(entry.get('changes')):
            value = change.get('value')
            if change.get('field') != 'messages' or not isinstance(value, dict):
                continue
            metadata = value.get('metadata')
            if (value.get('messaging_product') != 'whatsapp' or not isinstance(metadata, dict)
                    or metadata.get('phone_number_id') != expected_phone_id):
                continue
            for message in objects(value.get('messages')):
                text = message.get('text')
                phone = message.get('from')
                body = text.get('body') if isinstance(text, dict) else None
                if (message.get('type') == 'text' and isinstance(phone, str) and phone.strip()
                        and isinstance(body, str) and body.strip()):
                    message_id = message.get('id')
                    found.append(IncomingText(phone, body,
                        message_id if isinstance(message_id, str) and message_id else None))
    return found

@app.post('/webhook')
async def receive_webhook(request: Request, background_tasks: BackgroundTasks):
    try:
        require_whatsapp_mode()
    except TransportError:
        raise HTTPException(503, 'Transporte deshabilitado.') from None
    if not _wa_app_secret:
        raise HTTPException(503, 'Firma de webhook no configurada.')
    raw = bytearray()
    async for chunk in request.stream():
        raw.extend(chunk)
        if len(raw) > MAX_WEBHOOK_BYTES:
            raise HTTPException(413, 'Payload demasiado grande.')
    expected = 'sha256=' + hmac.new(_wa_app_secret.encode(), raw, hashlib.sha256).hexdigest()
    if not hmac.compare_digest(expected.encode(), request.headers.get('x-hub-signature-256', '').encode()):
        transport_event('SIGNATURE_REJECTED')
        raise HTTPException(403, 'Firma inválida.')
    try:
        payload = json.loads(raw)
    except (ValueError, UnicodeError):
        raise HTTPException(400, 'JSON inválido.') from None
    messages = extract_text_messages(payload, config.whatsapp_phone_number_id)
    # Reserva todo el lote antes de aceptar; no generar ni enviar en el endpoint.
    jobs = []
    with _wa_guard:
        if len(_wa_pending) + len(messages) > 100:
            raise HTTPException(503, 'Capacidad temporal agotada.')
        for message in messages:
            key = transport_ref(message.message_id) if message.message_id else None
            if key and key in _wa_seen:
                transport_event('DUPLICATE_IGNORED', message_ref=key)
                continue
            job = uuid4().hex
            _wa_pending.add(job)
            if key:
                _wa_seen[key] = job
                while len(_wa_seen) > 2000:
                    _wa_seen.popitem(last=False)
            jobs.append((job, message))
    for job, message in jobs:
        transport_event('MESSAGE_ACCEPTED', job=job,
                        message_ref=transport_ref(message.message_id) if message.message_id else None)
        background_tasks.add_task(process_incoming_text, job, message)
    if not messages:
        transport_event('EVENT_IGNORED')
    return {'accepted': len(jobs)}  # 200 antes de ejecutar BackgroundTasks.


In [ ]:
# [CELDA 24] Adaptador y envío WhatsApp Cloud API
def send_whatsapp_text(phone, response_text):
    require_whatsapp_mode()  # ANTES de construir el cliente o realizar cualquier HTTP.
    if not re.fullmatch(r'[0-9]+', config.whatsapp_phone_number_id):
        raise TransportError('Phone Number ID inválido.')
    if not re.fullmatch(r'v[0-9]+\.0', WHATSAPP_GRAPH_VERSION):
        raise TransportError('Versión Graph inválida.')
    if not isinstance(response_text, str) or not response_text.strip() or len(response_text) > 4096:
        raise TransportError('Texto vacío o superior al límite de WhatsApp; no se trunca.')
    # Reutiliza la normalización central. El núcleo ya devuelve este teléfono normalizado.
    recipient = normalize_phone(phone).lstrip('+')
    url = f'https://graph.facebook.com/{WHATSAPP_GRAPH_VERSION}/{config.whatsapp_phone_number_id}/messages'
    payload = {'messaging_product': 'whatsapp', 'recipient_type': 'individual',
               'to': recipient, 'type': 'text',
               'text': {'preview_url': False, 'body': response_text}}
    try:
        with httpx.Client(timeout=WHATSAPP_HTTP_TIMEOUT, follow_redirects=False,
                          trust_env=False) as sender:
            response = sender.post(url, headers={'Authorization': f'Bearer {config.whatsapp_access_token}'},
                                   json=payload)
            response.raise_for_status()
            data = response.json()
        messages = data.get('messages') if isinstance(data, dict) else None
        if (not isinstance(messages, list) or not messages or not isinstance(messages[0], dict)
                or not isinstance(messages[0].get('id'), str) or not messages[0]['id']):
            raise TransportError('Meta no confirmó la aceptación del mensaje.')
        return messages[0]['id']  # Aceptación NO equivale a entrega al dispositivo.
    except httpx.HTTPStatusError as error:
        # No propagar URL, cuerpo de error, request ni headers de la librería.
        raise TransportError(f'Meta HTTP {error.response.status_code}.') from None
    except (httpx.RequestError, ValueError):
        raise TransportError('Fallo de red o respuesta JSON inválida de Meta.') from None


def process_incoming_text(job, incoming):
    # BackgroundTask síncrona: FastAPI la ejecuta fuera del event loop.
    try:
        require_whatsapp_mode()
        result = process_message(incoming.phone, incoming.text)  # CoreResult existente.
        own = [e for e in diagnostics.snapshot() if e.run_id == result.run_id]
        complete = any(e.event == EventKind.RUN_STARTED for e in own) and any(
            e.event == EventKind.RUN_FINISHED for e in own)
        transport_event('CORE_COMPLETED', job=job, run_id=result.run_id,
            phone_ref=transport_ref(result.normalized_phone), route=result.route.value,
            history_count=result.history_count, history_trimmed=result.history_trimmed,
            conversation_ref=transport_ref(result.conversation_id) if result.conversation_id else None,
            llm_requests=sum(e.event == EventKind.LLM_REQUEST_STARTED for e in own) if complete else None)
        message_id = send_whatsapp_text(result.normalized_phone, result.response_text)
        transport_event('SEND_ACCEPTED', job=job, message_ref=transport_ref(message_id))
    except Exception as error:
        # El webhook ya respondió: no hay HTTP tardío ni traceback con datos personales.
        transport_event('BACKGROUND_FAILED', job=job, error_type=type(error).__name__)
    finally:
        with _wa_guard:
            _wa_pending.discard(job)


In [ ]:
# [CELDA 25] process_message()
# Núcleo compartido por pruebas locales y futuro transporte.
def process_message(phone_number: str, message: str) -> CoreResult:
    if not isinstance(message, str):
        raise TypeError("El mensaje debe ser string.")
    message = message.strip()
    if not message:
        raise ValueError("El mensaje no puede estar vacío.")
    phone = normalize_phone(phone_number)
    run_id = new_run_id()
    conversation_id = None
    lock_id = None
    route = None

    def emit(event, **fields):
        diagnostics.record(event, run_id=run_id, normalized_phone=phone,
                           conversation_id=conversation_id, lock_id=lock_id,
                           route=route, **fields)

    emit(EventKind.RUN_STARTED)
    try:
        # Antes del lock: no lee ni modifica conversaciones. Siempre None, incluso
        # si hubiera una previa: este recorrido no observa su ID ni su historial.
        route = select_route(message)
        if route != Route.LLM:
            emit(EventKind.ROUTE_SELECTED)
            emit(EventKind.LLM_SKIPPED)
            fixed = (VETERINARY_EMERGENCY_RESPONSE if route == Route.VETERINARY_EMERGENCY
                     else HUMAN_ESCALATION_RESPONSE)
            result = CoreResult(fixed, phone, None, route, None, run_id)
        else:
            route = Route.LLM
            emit(EventKind.ROUTE_SELECTED)
            phone_lock, lock_id = get_phone_lock(phone)
            emit(EventKind.LOCK_WAIT)
            phone_lock.acquire()
            try:
                emit(EventKind.LOCK_ACQUIRED)
                conversation_id = get_or_create_conversation(phone)
                conversation_id, _, trimmed_before = enforce_history_limit(phone, conversation_id, emit)
                text = generate_response(message, conversation_id, emit)
                conversation_id, history_count, trimmed_after = enforce_history_limit(phone, conversation_id, emit)
                result = CoreResult(text, phone, conversation_id, route, history_count, run_id,
                                    history_trimmed=trimmed_before or trimmed_after)
            finally:
                diagnostics.release_phone_lock(
                    phone_lock, run_id=run_id, normalized_phone=phone,
                    conversation_id=_conversation_ids.get(phone), lock_id=lock_id, route=route)
        emit(EventKind.RUN_FINISHED)
        return result
    except Exception:
        emit(EventKind.RUN_FAILED)
        raise

<sub>CELDA 26</sub>

### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [ ]:
# [CELDA 27] Servidor local y túnel ngrok reutilizable
def start_whatsapp_transport():
    global _wa_server, _wa_thread, _wa_tunnel, _wa_ngrok_config
    global _wa_app_secret, _wa_previous_diagnostics
    require_whatsapp_mode()
    if _wa_thread is not None and _wa_thread.is_alive():
        if _wa_server.should_exit or _wa_tunnel is None:
            raise TransportError('Servidor cerrando/incompleto: finalizar CELDA 34 antes de reiniciar.')
        print('CALLBACK_URL:', _wa_tunnel.public_url + '/webhook')
        return
    if _wa_tunnel is not None:
        raise TransportError('Túnel anterior pendiente de cierre: ejecutar CELDA 34.')
    if not callable(globals().get('process_message')) or globals().get('client') is None:
        raise TransportError('Preparar primero Stack y el núcleo compartido.')
    from google.colab import userdata
    try:
        app_secret = userdata.get('WHATSAPP_APP_SECRET')
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        app_secret = None
    if not isinstance(app_secret, str) or not app_secret.strip():
        raise TransportError('Configurar/autorizar WHATSAPP_APP_SECRET para verificar POST firmados.')
    import uvicorn
    import socket
    from pyngrok import ngrok, conf
    # Reservar el puerto antes del thread: fallo explícito si otro proceso lo ocupa.
    sock = socket.socket()
    try:
        sock.bind(('127.0.0.1', WEBHOOK_PORT))
    except OSError:
        sock.close()
        raise TransportError('Puerto del webhook ocupado; no se inicia otro servidor.') from None
    _wa_app_secret = app_secret.strip()
    _wa_previous_diagnostics = diagnostics.enabled
    diagnostics.enabled = True  # Cambiar solo sin pruebas locales/workers activos.
    _wa_server = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=WEBHOOK_PORT,
        access_log=False, log_level='critical', ws='none'))
    _wa_thread = threading.Thread(target=_wa_server.run, kwargs={'sockets': [sock]}, daemon=True)
    try:
        _wa_thread.start()
        deadline = time.monotonic() + 15
        while not _wa_server.started:
            if not _wa_thread.is_alive() or time.monotonic() >= deadline:
                raise TransportError('Servidor no alcanzó estado listo.')
            time.sleep(0.05)  # Solo readiness, no altera generación ni locks.
        # Sin guardar authtoken en archivo; sin inspector que almacene payloads.
        _wa_ngrok_config = conf.PyngrokConfig(auth_token=config.ngrok_auth_token,
                                             monitor_thread=False, max_logs=0)
        _wa_tunnel = ngrok.connect(addr=WEBHOOK_PORT, proto='http', bind_tls=True,
                                  inspect=False, pyngrok_config=_wa_ngrok_config)
    except Exception:
        _wa_server.should_exit = True
        _wa_thread.join(timeout=15)
        sock.close()
        diagnostics.enabled = _wa_previous_diagnostics
        _wa_app_secret = None
        raise TransportError('No se pudo iniciar servidor/túnel; revisar puerto, Secret y cuenta ngrok.') from None
    print('CALLBACK_URL:', _wa_tunnel.public_url + '/webhook')
    print('Usar WHATSAPP_VERIFY_TOKEN desde Secrets en Meta; suscribir messages.')
    print('TRANSPORT: IMPLEMENTED — PENDING META VALIDATION')


def stop_whatsapp_transport():
    global _wa_tunnel, _wa_app_secret
    if _wa_tunnel is not None:
        from pyngrok import ngrok
        try:
            ngrok.disconnect(_wa_tunnel.public_url, pyngrok_config=_wa_ngrok_config)
        except Exception:
            raise TransportError('No se confirmó cierre del túnel; reintentar sin reiniciar servidor.') from None
        _wa_tunnel = None
    if _wa_server is not None:
        _wa_server.should_exit = True
    if _wa_thread is not None and _wa_thread.is_alive():
        _wa_thread.join(timeout=5)
        if _wa_thread.is_alive():
            print('CLOSING: esperando BackgroundTasks; reejecutar cierre al terminar.')
            return
    if _wa_previous_diagnostics is not None:
        diagnostics.enabled = _wa_previous_diagnostics
    _wa_app_secret = None
    print('TRANSPORT STOPPED: túnel desconectado; servidor detenido.')

if MODE == WHATSAPP_MODE:
    start_whatsapp_transport()
else:
    print('LOCAL_TEST_MODE: servidor y ngrok no se inician.')


<sub>CELDA 28</sub>

El flujo conversacional local se valida en las pruebas ejecutables 1–6 (celdas 72–77): memoria, aislamiento, escalamiento, urgencia, recorte y concurrencia. Stack recupera el contexto mediante conversation=ID. Los espacios de transporte que siguen permanecen pendientes; no es necesario Meta para estas pruebas.

In [ ]:
# [CELDA 29] Demostración externa: primer mensaje
# Enviar MANUALMENTE desde WhatsApp al número habilitado en Meta; no desde Python.
print('PENDING META VALIDATION — mensaje 1: Mi mascota se llama Nacarina; quiero una consulta general.')


In [ ]:
# [CELDA 30] Demostración externa: memoria, recorte y urgencia
print('PENDING META VALIDATION — enviar uno a uno y esperar cada respuesta:')
for number, text in enumerate([
    '¿Cómo se llama mi mascota?',
    'Nacarina necesita una consulta general. ¿Qué información debo llevar?',
    '¿Cómo se llama mi mascota y qué consulta he mencionado?',
    '¿Qué información debo llevar a esa consulta?',
    'Mi mascota no puede respirar y quiero hablar con una persona.'
], start=2):
    print(number, text)
print('Confirmar recepción en WhatsApp. Mensaje 4: recorte; 5: continuidad; 6: urgencia sin LLM.')


In [ ]:
# [CELDA 31] Inspección segura del flujo externo
def show_transport_evidence():
    rows = transport_snapshot()
    if not rows:
        print('PENDING META VALIDATION — sin eventos externos.')
    for row in rows:
        print(row)  # Solo campos permitidos; sin texto, teléfono, ID remoto ni token.
    print('SEND_ACCEPTED solo confirma aceptación API; verificar entrega en WhatsApp.')
    print('Sin PASS automático: comprobar manualmente verificación, respuestas y eventos.')

show_transport_evidence()


In [ ]:
# [CELDA 32] Mensajes aceptados sin contenido sensible
for row in transport_snapshot():
    if row['event'] in {'MESSAGE_ACCEPTED', 'EVENT_IGNORED', 'BACKGROUND_FAILED', 'SEND_ACCEPTED'}:
        print(row)
print('PENDING META VALIDATION si aún no se enviaron mensajes reales.')


In [ ]:
# [CELDA 33] Estado de historial observado por el núcleo
# CoreResult refleja la inspección REAL de Stack realizada bajo el lock del núcleo.
# No consultar/mostrar mensajes privados ni duplicar su memoria aquí.
for row in transport_snapshot():
    if row['event'] == 'CORE_COMPLETED':
        print(row)
print('Comprobar history_trimmed=True, history_count<=6 y continuidad; urgencia con llm_requests=0.')
print('llm_requests=None significa evidencia incompleta: NO afirmar cero.')


In [ ]:
# [CELDA 34] Cerrar túnel y servidor del Hands-On
stop_whatsapp_transport()


<sub>CELDA 35</sub>

## **CHALLENGE: AGENTE DE UNA VETERINARIA**

<sub>CELDA 36</sub>

Una vez visto el ***Hands-On: Diseño de Agentes Conversacionales***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

<sub>CELDA 37</sub>

Se construirá un agente similar, pero para una veterinaria: además de la regla de escalamiento a una persona, se agrega una regla de urgencia médica, y un límite al historial que se conserva en la conversación de Llama Stack.

<sub>CELDA 38</sub>

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

<sub>CELDA 39</sub>

### **INSTRUCCIONES:**

<sub>CELDA 40</sub>

**1. Prepara el entorno:** Instala y arranca Ollama en Colab, descarga el modelo Llama 3.1 8B, instala y levanta **Llama Stack** con Ollama registrado, y carga las credenciales de WhatsApp desde **Colab Secrets**.

In [ ]:
# [CELDA 41] Challenge: reutilizar instalación de Ollama
# Instalar Ollama (requiere zstd, que Colab no trae por defecto)
# Fase 1: ejecutar primero el entorno Hands-On; reutilizar sus funciones.
install_ollama()

In [ ]:
# [CELDA 42] Challenge: preparar servicio y modelo Ollama
# Iniciar el servidor de Ollama en segundo plano
# Reutilizar servicio/modelo; no iniciar segunda instancia.
prepare_ollama_model()

In [ ]:
# [CELDA 43] Challenge: reutilizar instalación de dependencias
# Instalar dependencias
# Mismo entorno aislado y versiones del Hands-On.
install_stack_dependencies()

In [ ]:
# [CELDA 44] Challenge: preparar cliente Llama Stack
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado
# Reutilizar servicio verificando readiness y APIs.
client = prepare_stack_client()

In [ ]:
# [CELDA 45] Challenge: cargar configuración del modo seleccionado
# Leer credenciales desde Colab Secrets
# MODE se define en el Hands-On; modo local no consulta Secrets.
config = load_config(MODE)

<sub>CELDA 46</sub>

**2. Construye la memoria de conversación:** Crea una conversación de **Llama Stack** por número de teléfono (`client.conversations.create()`) con una memoria de respuesta.

In [ ]:
# [CELDA 47] Referencia a memoria y política de historial
# Memoria única en la celda 17; política MAX_HISTORY_ITEMS y recorte real en la 19.
# Stack 0.7.3 requiere rotar además de borrar items: ver explicación en la celda 66.

In [ ]:
# [CELDA 48] Referencia a generación y recorte
# Generación única en la celda 19: generate_response / enforce_history_limit.
# El núcleo (25) comprueba y limita el historial antes y después de generar.

<sub>CELDA 49</sub>

**3. Agrega la regla fija y protege el procesamiento por número:**

   * Cuando el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento.
   * Igual que en el Hands-On, usa un lock por número de teléfono para que dos mensajes seguidos del mismo usuario no disparen dos llamadas en paralelo sobre la misma conversación.

In [ ]:
# [CELDA 50] Referencia a locks e instrumentación de concurrencia
# Locks únicos en la celda 17. El núcleo (25) protege generación y recorte.
# Recorder con secuencia thread-safe; prueba real de threads en la celda 77.

In [ ]:
# [CELDA 51] Referencia al núcleo y prioridad de reglas
# Núcleo único en la celda 25: process_message; select_route en la 19.
# Prioridad propia: VETERINARY_EMERGENCY > HUMAN_ESCALATION > LLM.

<sub>CELDA 52</sub>

**4. Define los endpoints del webhook:** usa `BackgroundTasks` para responder de inmediato a Meta, delegando el procesamiento real a la función anterior.

In [ ]:
# [CELDA 53] Servidor compartido y pruebas deterministas del transporte
# Ejecutar pruebas solo en LOCAL_TEST_MODE, con servidor detenido.
# Dobles exclusivamente del transporte: NO validan Meta ni reemplazan pruebas 1–6.
def test_transport_adapter():
    import asyncio
    from unittest.mock import patch
    from types import SimpleNamespace
    from starlette.requests import Request as TestRequest
    from fastapi.testclient import TestClient
    if MODE != LOCAL_TEST_MODE or (_wa_thread is not None and _wa_thread.is_alive()):
        raise RuntimeError('Ejecutar pruebas de transporte en LOCAL_TEST_MODE y sin servidor.')
    # Colab ya tiene event loop: ejecutar coroutines de fixtures en un thread separado.
    from concurrent.futures import ThreadPoolExecutor
    def run_fixture(coroutine):
        with ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(asyncio.run, coroutine).result(timeout=15)
    ns = globals()
    fake_config = AppConfig(WHATSAPP_MODE, 'synthetic-access', '1234567890', 'synthetic-verify', 'synthetic-ngrok')
    payload = {'object': 'whatsapp_business_account', 'entry': [{'changes': [{
        'field': 'messages', 'value': {'messaging_product': 'whatsapp',
        'metadata': {'phone_number_id': '1234567890'},
        'messages': [{'from': '15550000001', 'id': 'synthetic-message',
                      'type': 'text', 'text': {'body': 'Hola'}}]}}]}]}
    raw = json.dumps(payload).encode()
    signature = 'sha256=' + hmac.new(b'synthetic-app', raw, hashlib.sha256).hexdigest()
    synthetic = {'MODE': WHATSAPP_MODE, 'config': fake_config, '_wa_app_secret': 'synthetic-app',
                 '_wa_events': deque(maxlen=500), '_wa_pending': set(), '_wa_seen': OrderedDict()}
    checked = []
    print('FIXTURES ONLY - URLs y resultados siguientes son sintéticos; no se contacta Meta/ngrok.')
    # Guarda: ningún cliente HTTP debe siquiera construirse.
    with patch.object(httpx, 'Client', side_effect=AssertionError('HTTP prohibido')):
        try:
            send_whatsapp_text('+15550000001', 'Hola')
        except TransportError:
            pass
        else:
            raise AssertionError('Guarda local ausente.')
    with TestClient(app) as web:
        assert web.post('/webhook', json={}).status_code == 503
        assert web.get('/webhook').status_code == 503
    checked.append('LOCAL_SEND_AND_WEBHOOK_BLOCKED')
    assert len(extract_text_messages(payload, '1234567890')) == 1
    assert extract_text_messages(payload, 'different') == []
    for ignored in [None, [], {}, {'object': 'whatsapp_business_account', 'entry': [None]}]:
        assert extract_text_messages(ignored, '1234567890') == []
    variant = json.loads(raw)
    value = variant['entry'][0]['changes'][0]['value']
    value['messages'] += [None, {'type': 'image'}, {'type': 'text', 'from': '15550000001', 'text': {'body': ' '}}]
    assert len(extract_text_messages(variant, '1234567890')) == 1
    value.pop('messages'); value['statuses'] = [{'status': 'delivered'}]
    assert extract_text_messages(variant, '1234567890') == []
    checked.append('PARSING_AND_IGNORED_EVENTS')
    with patch.dict(ns, synthetic):
        with TestClient(app) as web:
            assert web.get('/webhook', params={'hub.mode': 'subscribe', 'hub.verify_token': 'synthetic-verify', 'hub.challenge': '42'}).text == '42'
            assert web.get('/webhook', params={'hub.mode': 'subscribe', 'hub.verify_token': 'wrong', 'hub.challenge': '42'}).status_code == 403
            assert web.post('/webhook', content=raw).status_code == 403
            bad = b'not-json'
            bad_sig = 'sha256=' + hmac.new(b'synthetic-app', bad, hashlib.sha256).hexdigest()
            assert web.post('/webhook', content=bad, headers={'x-hub-signature-256': bad_sig}).status_code == 400
        checked.append('GET_AND_SIGNATURE_AND_BAD_JSON')
        # Endpoint directo: retorna ANTES de ejecutar la tarea; no interviene LLM.
        async def delegate():
            async def receive():
                return {'type': 'http.request', 'body': raw, 'more_body': False}
            request = TestRequest({'type': 'http', 'method': 'POST', 'path': '/webhook',
                'headers': [(b'x-hub-signature-256', signature.encode())]}, receive)
            bg = BackgroundTasks()
            answer = await receive_webhook(request, bg)
            assert answer == {'accepted': 1} and len(bg.tasks) == 1
            assert bg.tasks[0].func is process_incoming_text
            return bg
        bg = run_fixture(delegate())
        assert not any(row['event'] == 'CORE_COMPLETED' for row in transport_snapshot())
        with TestClient(app) as web:
            assert web.post('/webhook', content=raw, headers={'x-hub-signature-256': signature}).json() == {'accepted': 0}
        checked.append('DELEGATION_BEFORE_PROCESSING_AND_DEDUP')
        # Protocolo ASGI real, sin socket: el cuerpo HTTP sale antes de la tarea.
        async def asgi_ack_before_task():
            second = json.loads(raw)
            second['entry'][0]['changes'][0]['value']['messages'][0]['id'] = 'synthetic-second'
            body = json.dumps(second).encode()
            signed = 'sha256=' + hmac.new(b'synthetic-app', body, hashlib.sha256).hexdigest()
            sent, called = [], []
            async def receive():
                return {'type': 'http.request', 'body': body, 'more_body': False}
            async def send(event):
                sent.append(event)
            def fixture_task(job, message):
                assert any(e['type'] == 'http.response.body' and not e.get('more_body', False) for e in sent)
                called.append(message)
                _wa_pending.discard(job)
            scope = {'type': 'http', 'asgi': {'version': '3.0'}, 'http_version': '1.1',
                     'method': 'POST', 'scheme': 'http', 'path': '/webhook', 'raw_path': b'/webhook',
                     'query_string': b'', 'root_path': '', 'server': ('testserver', 80),
                     'client': ('127.0.0.1', 1), 'headers': [(b'x-hub-signature-256', signed.encode())]}
            with patch.dict(ns, {'process_incoming_text': fixture_task}):
                await app(scope, receive, send)
            assert len(called) == 1
            assert next(e['status'] for e in sent if e['type'] == 'http.response.start') == 200
        run_fixture(asgi_ack_before_task())
        checked.append('ASGI_RESPONSE_SENT_BEFORE_BACKGROUND_TASK')
        fake_result = SimpleNamespace(normalized_phone='+15550000001', response_text='respuesta de fixture',
            run_id='fixture-run', route=Route.HUMAN_ESCALATION, history_count=None,
            history_trimmed=None, conversation_id=None)
        def fixture_core(phone, text):
            assert phone == '15550000001' and text == 'Hola'
            return fake_result
        with patch.dict(ns, {'process_message': fixture_core}), patch.dict(ns, {'send_whatsapp_text': lambda phone, text: 'synthetic-out'}):
            run_fixture(bg())
        assert any(row['event'] == 'SEND_ACCEPTED' for row in transport_snapshot())
        def fail(*args):
            raise RuntimeError('synthetic-sensitive-detail')
        with patch.dict(ns, {'process_message': fail}):
            process_incoming_text('failure-fixture', IncomingText('15550000001', 'Hola'))
        assert transport_snapshot()[-1]['event'] == 'BACKGROUND_FAILED'
        assert 'synthetic-sensitive-detail' not in str(transport_snapshot())
        checked.append('BACKGROUND_SUCCESS_AND_SAFE_FAILURE')
        real_client = httpx.Client
        requests = []
        def respond(request):
            requests.append(request)
            assert request.url == 'https://graph.facebook.com/v23.0/1234567890/messages'
            assert request.headers['authorization'] == 'Bearer synthetic-access'
            assert json.loads(request.content) == {'messaging_product': 'whatsapp', 'recipient_type': 'individual',
                'to': '15550000001', 'type': 'text', 'text': {'preview_url': False, 'body': 'Hola'}}
            return httpx.Response(200, json={'messages': [{'id': 'synthetic-out'}]})
        def factory(**kwargs):
            assert kwargs['timeout'].read == 30 and kwargs['follow_redirects'] is False
            return real_client(transport=httpx.MockTransport(respond), **kwargs)
        with patch.object(httpx, 'Client', side_effect=factory):
            assert send_whatsapp_text('+15550000001', 'Hola') == 'synthetic-out'
        assert len(requests) == 1
        for status, data in [(401, {'error': {'message': 'synthetic-sensitive-detail'}}), (200, {})]:
            def failing_factory(**kwargs):
                return real_client(transport=httpx.MockTransport(lambda req: httpx.Response(status, json=data)), **kwargs)
            with patch.object(httpx, 'Client', side_effect=failing_factory):
                try:
                    send_whatsapp_text('+15550000001', 'Hola')
                except TransportError as error:
                    assert 'synthetic-sensitive-detail' not in str(error) and 'synthetic-access' not in str(error)
                else:
                    raise AssertionError('Error de envío no detectado.')
        def timeout_response(request):
            raise httpx.ReadTimeout('synthetic-sensitive-detail', request=request)
        with patch.object(httpx, 'Client', side_effect=lambda **kwargs: real_client(
                transport=httpx.MockTransport(timeout_response), **kwargs)):
            try:
                send_whatsapp_text('+15550000001', 'Hola')
            except TransportError as error:
                assert 'synthetic-sensitive-detail' not in str(error)
            else:
                raise AssertionError('Timeout no detectado.')
        checked.append('REQUEST_CONSTRUCTION_AND_HTTP_FAILURES_AND_TIMEOUT')
    # Solo dobles del ciclo de vida; no se abre túnel ni servidor de red.
    from pyngrok import ngrok
    fake_thread = SimpleNamespace(is_alive=lambda: True, join=lambda timeout: None)
    with patch.dict(ns, {**synthetic, '_wa_thread': fake_thread,
                        '_wa_server': SimpleNamespace(should_exit=False),
                        '_wa_tunnel': SimpleNamespace(public_url='https://example.invalid'),
                        '_wa_ngrok_config': None}):
        with patch.object(ngrok, 'connect', side_effect=AssertionError('No abrir otro túnel')), patch.object(
                ngrok, 'disconnect') as disconnect:
            start_whatsapp_transport()  # Reutiliza el servidor vivo.
            stop_whatsapp_transport()   # Desconecta y espera, sin matar tareas.
            disconnect.assert_called_once_with('https://example.invalid', pyngrok_config=None)
            assert _wa_server.should_exit and _wa_tunnel is None
            assert _wa_app_secret == 'synthetic-app'  # Tareas aún vivas: no retirar credenciales.
            fake_thread.is_alive = lambda: False
            stop_whatsapp_transport()
            assert _wa_app_secret is None
    checked.append('REUSE_TUNNEL_AND_GRACEFUL_CLOSE_FIXTURES')
    print('TRANSPORT UNIT TESTS: PASSED', checked)
    print('Solo fixtures; integración externa: PENDING META VALIDATION')

if MODE == LOCAL_TEST_MODE:
    test_transport_adapter()
else:
    print('Servidor compartido de CELDAS 14, 22–27; fixtures omitidas en WHATSAPP_MODE.')


In [ ]:
# [CELDA 54] Verificación de la normalización compartida
# Definir función de normalización de número
# La única implementación es normalize_phone, definida en Hands-On.
# Webhook y pruebas locales deberán llamar a esa misma función.
assert callable(normalize_phone)

In [ ]:
# [CELDA 55] Reutilizar verificación del webhook
assert callable(verify_webhook)
print('GET /webhook compartido — PENDING META VALIDATION')


In [ ]:
# [CELDA 56] Reutilizar recepción y BackgroundTasks
assert callable(receive_webhook) and callable(process_incoming_text)
print('POST /webhook compartido — PENDING META VALIDATION')


In [ ]:
# [CELDA 57] Reutilizar envío WhatsApp
assert callable(send_whatsapp_text)
print('Envío compartido; bloqueado en LOCAL_TEST_MODE — PENDING META VALIDATION')


<sub>CELDA 58</sub>

**5. Expón el servidor con ngrok y prueba el flujo completo:**

   * Manda una conversación de al menos 4 mensajes y confirma que el historial se recorta correctamente.
   * Manda un mensaje con una palabra de urgencia y confirma que dispara la regla fija en vez de una respuesta de Llama.

In [ ]:
# [CELDA 59] Iniciar el mismo transporte para el Challenge
if MODE == WHATSAPP_MODE:
    start_whatsapp_transport()
else:
    print('LOCAL_TEST_MODE: sin servidor público ni ngrok.')


In [ ]:
# [CELDA 60] Demostración externa del Challenge: primer mensaje
print('PENDING META VALIDATION — seguir CELDA 29 desde WhatsApp; mismo núcleo y transporte.')


In [ ]:
# [CELDA 61] Demostración externa del Challenge: conversación y urgencia
print('PENDING META VALIDATION — seguir la secuencia completa de CELDA 30, esperando cada respuesta.')


In [ ]:
# [CELDA 62] Verificar resultados externos del Challenge
show_transport_evidence()


In [ ]:
# [CELDA 63] Inspeccionar recepción externa del Challenge
for row in transport_snapshot():
    if row['event'] in {'MESSAGE_ACCEPTED', 'BACKGROUND_FAILED', 'SEND_ACCEPTED'}:
        print(row)


In [ ]:
# [CELDA 64] Inspeccionar historial y bypass externos del Challenge
for row in transport_snapshot():
    if row['event'] == 'CORE_COMPLETED':
        print(row)
print('PENDING META VALIDATION hasta comprobar recorte, continuidad y urgencia con cero LLM.')


In [ ]:
# [CELDA 65] Cerrar el transporte compartido del Challenge
stop_whatsapp_transport()


<sub>CELDA 66</sub>

## SOPORTE DE IMPLEMENTACIÓN — FASES 1–3

Estas celdas auxiliares complementan el núcleo implementado en el Hands-On.
Ejecutar en orden el entorno Hands-On (6, 8, 9, 11, 12) y las celdas de lógica (15, 17, 19, 25).
El Challenge reutiliza esas funciones; no es una segunda instalación independiente.
Se requiere Colab Linux con Python >=3.12, espacio para Llama 3.1 8B y recursos suficientes.
Una GPU es recomendable; no se garantiza velocidad en CPU. La descarga puede tardar varios minutos.
El runtime y el almacenamiento nativo de Stack son efímeros al eliminar la sesión Colab.

**Modos y Secrets:** LOCAL_TEST_MODE no consulta Meta ni arranca ngrok.
WHATSAPP_MODE exige WHATSAPP_ACCESS_TOKEN, WHATSAPP_PHONE_NUMBER_ID,
WHATSAPP_VERIFY_TOKEN y NGROK_AUTH_TOKEN en Colab Secrets, con acceso autorizado.
Son nombres elegidos por esta implementación. No pegar valores ni números personales en celdas/outputs.
Configurar Secrets no acredita integración. Transporte: IMPLEMENTED — PENDING META VALIDATION.
Demostración externa: PENDING META VALIDATION. POST requiere además WHATSAPP_APP_SECRET (App Secret de Meta, no verify token); se lee solo al iniciar transporte WhatsApp.

**Versiones:** servidor/API/cliente Llama Stack 0.7.3; proveedor Python Ollama 0.6.3.
Servidor en entorno virtual aislado; cliente en el kernel. Se limita MCP a >=1.23.0,<2: MCP 2 elimina McpError, importado por Stack 0.7.3.
Las demás dependencias transitivas no son un lockfile.
Se comprueban métodos instalados y rutas del servidor antes de declarar el entorno listo.
Núcleo local: VALIDATED IN COLAB (pruebas 1–6; evidencia separada, sin copiar outputs). La persistencia es la nativa de Llama Stack:
no se consulta directamente una base de datos ni se reemplaza conversations por memoria propia.

Referencias:
[paquete Llama Stack](https://pypi.org/project/llama-stack/0.7.3/),
[cliente](https://pypi.org/project/llama-stack-client/0.7.3/),
[Ollama Linux](https://docs.ollama.com/linux),
[modelo llama3.1:8b](https://ollama.com/library/llama3.1:8b).

**Arquitectura del núcleo y transporte futuro:** celda local → process_message → texto/diagnóstico;
WhatsApp → webhook/tarea de fondo → la misma process_message → envío WhatsApp.
Normalización central; envío Meta fuera del núcleo. Conversaciones por teléfono,
HUMAN_ESCALATION, VETERINARY_EMERGENCY, recorte, memoria, locks y procesamiento implementados; transporte IMPLEMENTED — PENDING META VALIDATION.

**API 0.7.3 inspeccionada (wheel cliente y servidor):**

| Operación | Firma relevante en client | Uso |
|---|---|---|
| Crear conversación | conversations.create(*, items=..., metadata=...) | Crear y rotar con seis items |
| Recuperar conversación | conversations.retrieve(conversation_id) | Disponible, no necesaria |
| Listar items | conversations.items.list(conversation_id, *, after=..., limit=..., order=...) | Inspección orden asc |
| Recuperar item | conversations.items.get(item_id, *, conversation_id=...) | Disponible, no necesaria |
| Borrar item | conversations.items.delete(item_id, *, conversation_id=...) | Borrado remoto de antiguos |
| Borrar conversación | conversations.delete(conversation_id) | Disponible, no utilizada |
| Agregar items | conversations.items.create(conversation_id, *, items=...) | Disponible (hasta 20), no necesaria |
| Reconstrucción | conversations.create(items=payloads) | Contexto nativo en ID nuevo, sin IDs de items antiguos |

No hay operación de clonado automático utilizada. Los payloads transitorios conservan
rol/contenido de los mensajes textuales; se rechazan items que no puedan transferirse
sin perder contexto. No se guardan mensajes en el índice local ni se reenvía una lista
Python como contexto de responses.create: la memoria sigue en Stack.

**Limitaciones reales encontradas en el servidor 0.7.3:**
- `core/conversations/conversations.py:list_items` corta por limit, ignora after y
  devuelve has_more=False. La paginación del cliente no corrige esto. Se solicita
  limit=100 y se rechaza una página llena; no se inventa un total. El flujo habitual
  mantiene seis items más los del turno actual. La política usa el orden asc provisto
  por Stack; los timestamps del servidor pueden empatar y no se promete orden más fino.
- `providers/inline/responses/builtin/responses/openai_responses.py` recupera
  `responses_store.get_conversation_messages` como contexto cuando ya existe.
  `items.delete` solo borra de conversation_items: no invalida ese contexto interno.
  Por eso la preferencia de borrado se complementa con rotación real: crear con los
  seis recientes, verificar contenido/orden e IDs nuevos, cambiar el índice y borrar
  los items antiguos del ID anterior. La nueva conversación no hereda la caché del ID
  anterior y su siguiente generación se reconstruye desde los items transferidos.
- `conversations.delete` borra la cabecera en esa implementación; no se utiliza ni
  se promete purgar items, responses o conversation_messages de IDs abandonados.

**Política propia:** MAX_HISTORY_ITEMS=6, unidad item de la API, no turno.
El notebook exige historial limitado, pero NO especifica seis: lo elegimos para
mostrar el recorte con pocas llamadas y reducir costo. Se comprueba antes y después
de generar, siempre bajo el lock del teléfono. Durante la generación puede superarse
el límite; solo se devuelve un CoreResult LLM exitoso tras verificar el estado final.
Una lectura incompleta o fallo de recorte se propaga, no se declara éxito con None.
Las reglas no inspeccionan historial: history_count/history_trimmed=None.
La copia nativa y el borrado no son una transacción: si falla antes de cambiar el
índice permanece el anterior; si falla después permanece el nuevo ya verificado.
Puede quedar una conversación candidata huérfana. No se promete rollback remoto.

**Reglas:** VETERINARY_EMERGENCY > HUMAN_ESCALATION > LLM es una DECISIÓN DE
IMPLEMENTACIÓN, NO una regla textual especificada por el notebook. La urgencia
prevalece sobre el texto administrativo. Los patrones explícitos están en la celda
19; no incluyen sangre, dolor, enfermo ni vómito aislados. Es una demostración por
presencia de frases: no interpreta negaciones ni historia clínica; puede activar
ante citas o menciones históricas. No diagnostica ni da instrucciones clínicas.

**Diagnóstico:** desactivado por defecto, acotado a 1000 eventos y protegido por lock.
Solo los teléfonos sintéticos de prueba aparecen normalizados; otros se correlacionan
con HMAC efímero. No se guardan mensajes, respuestas, excepciones completas ni Secrets.
Los contadores de prueba usan run_id y exigen eventos de inicio/fin para detectar
pérdida de evidencia por rotación. El cliente usa max_retries=0: un evento STARTED
representa un intento real de responses.create, incluidas solicitudes fallidas.
No cuenta operaciones de conversations ni asegura que un fallo de red llegó al modelo.

**Orden y alcance:** ejecutar las definiciones antes de las pruebas. Ambas reglas
se evalúan antes del lock y no observa ni toca conversaciones: ID e historial quedan
None. No envía avisos reales al equipo. Un fallo de generación se propaga y libera
el lock; no se reintenta automáticamente ni se promete rollback remoto.
Los locks solo coordinan este proceso/sesión, no múltiples kernels o workers.
Pruebas locales 1–6: VALIDATED IN COLAB. 1, 2, 5 y 6 con Stack real; 3 y 4 sin generaciones.

**Concurrencia:** cada recorder tiene una secuencia global creciente, asignada junto
con timestamp y append bajo RLock. Snapshot/clear usan el mismo guard; clear no
reinicia la secuencia. La liberación real del lock y LOCK_RELEASED ocurren bajo el
guard del recorder para impedir que otro LOCK_ACQUIRED se registre antes del release.
Ese guard solo serializa eventos, nunca llamadas de red. La prueba usa Barrier y
threads; verifica intervalos estructurados y locks distintos, sin exigir FIFO ni
inferir paralelismo del proveedor por duración. Reset/reejecución de definiciones y
cambios de diagnostics.enabled requieren núcleo y pruebas inactivos. No se soportan
resets deliberados durante procesamiento concurrente.

**Transporte compartido (sin cambiar el núcleo):** ejecutar CELDAS 14, 22, 23 y 24 para definiciones;
CELDA 27 inicia servidor/túnel solo en WHATSAPP_MODE. CELDA 53 prueba únicamente el adaptador con fixtures,
en modo local y servidor detenido. CELDAS 55–57 reutilizan; CELDA 59 inicia el mismo servidor de forma idempotente.
Nunca reejecutar definiciones del núcleo/transporte ni cambiar modo/Secrets con tareas activas.
CELDA 34 o 65 desconecta el túnel y drena tareas; si dice CLOSING, esperar y repetir cierre.
Para reiniciar tras cambios: cerrar, reejecutar definiciones de transporte y arrancar.
La deduplicación cubre los últimos 2000 IDs durante esta sesión, incluso fallidos; sin ID no se promete deduplicar.
No hay cola durable, reintentos automáticos, garantía exactly-once ni FIFO de entrega. Un fallo queda BACKGROUND_FAILED;
no se reenvía automáticamente porque un timeout puede ocurrir después de aceptar Meta el envío.
Los estados entrantes se ignoran: SEND_ACCEPTED prueba aceptación API, no entrega. Verificar en WhatsApp.
El inspector ngrok y access logs se desactivan para no almacenar cuerpos ni verify tokens. No mostrar Secrets.
La evidencia de transporte solo contiene referencias HMAC, rutas y conteos. llm_requests=None no significa cero.
El diagnóstico se habilita al arrancar y se restaura tras cerrar; no ejecutar simultáneamente pruebas locales.

**Demostración manual:** cambiar MODE a WHATSAPP_MODE en CELDA 06 y cargar los cuatro Secrets existentes,
más WHATSAPP_APP_SECRET para autenticar POST. Ejecutar definiciones compartidas y CELDA 27 (o 59).
Configurar en Meta la CALLBACK_URL impresa, verify token desde Secret y suscripción messages.
Seguir CELDAS 29–30 (Challenge: 60–61); esperar respuesta a cada mensaje.
Revisar CELDAS 31–33 o 62–64: VERIFY_ACCEPTED, MESSAGE_ACCEPTED, CORE_COMPLETED, SEND_ACCEPTED;
para urgencia exigir VETERINARY_EMERGENCY y llm_requests=0; para recorte history_trimmed=True y límite seis.
Comprobar manualmente recepción y contenido en WhatsApp. Cerrar con CELDA 34 o 65.
No se incorpora automáticamente evidencia del notebook (pruebas) ni se afirma validación Meta.

Fuentes del transporte: [payloads oficiales de Meta](https://www.postman.com/meta/whatsapp-business-platform/folder/vzaxn16/webhook-payload-reference),
[envío de texto](https://www.postman.com/meta/whatsapp-business-platform/request/8gvd47s/send-text-message),
[firma de webhook, SDK de Meta](https://whatsapp.github.io/WhatsApp-Nodejs-SDK/api-reference/webhooks/start/),
[BackgroundTasks](https://fastapi.tiangolo.com/tutorial/background-tasks/),
[pyngrok](https://pyngrok.readthedocs.io/en/stable/api.html).

Contrato Graph usado: [especificación oficial v23.0](https://github.com/facebook/openapi/blob/main/business-messaging-api_v23.0.yaml). La versión se configura en CELDA 14; no actualiza Stack.


In [ ]:
# [CELDA 67] Diagnóstico del entorno y registro de modelos
# Diagnóstico de entorno: solo estado no sensible, nunca configuración completa.
def diagnose_environment():
    ollama_ok = service_ready(OLLAMA_URL + "/api/tags")
    model_ok = False
    stack_ok = False
    if ollama_ok:
        model_ok = OLLAMA_MODEL in {
            m.get("name") for m in get_json(OLLAMA_URL + "/api/tags").get("models", [])}
    if service_ready(STACK_URL + "/v1/health") and globals().get("client") is not None:
        try:
            if get_json(STACK_URL + "/v1/version").get("version") == STACK_VERSION:
                require_registered_stack_model(client.models.list())
                stack_ok = True
        except ModelRegistryError as error:
            print("MODEL_REGISTRY .....", str(error))  # Solo diagnóstico seguro del helper.
            stack_ok = False
        except Exception as error:
            print("STACK_CHECK_ERROR ..", type(error).__name__)  # Sin payload/configuración.
            stack_ok = False
    ready = IN_COLAB and ollama_ok and model_ok and stack_ok
    rows = {
        "COLAB": "OK" if IN_COLAB else "NOT RUNNING IN COLAB",
        "OLLAMA": "OK" if ollama_ok else "NOT READY",
        "OLLAMA_MODEL": OLLAMA_MODEL,
        "LLAMA_STACK_MODEL": LLAMA_STACK_MODEL,
        "OLLAMA_MODEL_READY": "OK" if model_ok else "NOT READY",
        "MODEL_REGISTRATION": "OK" if stack_ok else "NOT VERIFIED",
        "LLAMA_STACK": "OK" if stack_ok else "NOT READY",
        "LOCAL_TEST_MODE": "ENVIRONMENT READY" if ready else "ENVIRONMENT NOT READY",
        "WHATSAPP_MODE": "SECRETS CONFIGURED; IMPLEMENTED — PENDING META VALIDATION" if config.transport_configured else "NOT CONFIGURED",
        "PROCESS_MESSAGE": "IMPLEMENTED; local tests VALIDATED IN COLAB",
    }
    for name, status in rows.items():
        print(f"{name:.<24} {status}")

diagnose_environment()

In [ ]:
# [CELDA 68] Reset local, conteo de solicitudes y presentación de resultados
# Utilidades locales. Ejecutar resets con pruebas inactivas, entre ejecuciones.
def reset_local_test_state(phones):
    phones = {normalize_phone(phone) for phone in phones}
    if not phones <= SYNTHETIC_PHONES:
        raise ValueError("Reset limitado a teléfonos sintéticos declarados.")
    for phone in sorted(phones):
        phone_lock, _ = get_phone_lock(phone)
        with phone_lock:
            _conversation_ids.pop(phone, None)
            diagnostics.clear(phones={phone})
    # Conservar locks evita crear dos locks para el mismo teléfono ante un waiter.
    # NO elimina conversaciones ni items en Stack. El próximo turno crea otro ID.


def llm_request_count(results):
    events = diagnostics.snapshot()
    run_ids = {result.run_id for result in results}
    for run_id in run_ids:
        kinds = [e.event for e in events if e.run_id == run_id]
        if EventKind.RUN_STARTED not in kinds or EventKind.RUN_FINISHED not in kinds:
            raise RuntimeError("Evidencia incompleta o diagnóstico desactivado.")
    return sum(e.event == EventKind.LLM_REQUEST_STARTED and e.run_id in run_ids for e in events)


def show_test_results(results):
    for result in results:
        assert result.normalized_phone in SYNTHETIC_PHONES
        print({"run_id": result.run_id, "normalized_phone": result.normalized_phone,
               "conversation_id": result.conversation_id, "route": result.route.value,
               "history_count": result.history_count, "history_trimmed": result.history_trimmed})
        print(result.response_text)
    print("LLM_REQUEST_STARTED:", llm_request_count(results))

<sub>CELDA 69</sub>

## PRUEBAS LOCALES — SIN META

Ejecutar las definiciones Hands-On y las utilidades antes de estas pruebas.
Las pruebas 1, 2, 5 y 6 invocan Stack y llama3.1:8b reales: requieren Colab listo.
Las pruebas 3 y 4 funcionan sin cliente; sus reglas evitan cualquier acceso a Stack.
No hay sustitutos ni resultados exitosos prefabricados. Los fallos se propagan.
Cada prueba limpia sus índices/diagnósticos locales con los workers inactivos,
conserva locks y restaura la activación previa del recorder. No elimina conversaciones.

1. **Memoria:** misma conversación y dato controlado, dos solicitudes.
2. **Aislamiento:** dos teléfonos/datos diferentes, cuatro solicitudes.
3. **Escalamiento:** respuesta fija, ID None, cero solicitudes y negativos.
4. **Urgencia:** positivos, negativos y prioridad sobre escalamiento; cero solicitudes.
5. **Recorte real:** seis items, borrado/rotación verificados y generación posterior.
6. **Concurrencia:** Barrier y threads con mismo teléfono y teléfonos diferentes.

Pruebas locales 1–6: VALIDATED IN COLAB; archivo (pruebas) usado exclusivamente como evidencia READ-ONLY.
No se identificó un bloqueo de API para la estrategia adaptada. Errores de servicios,
transferencia o verificación fallan explícitamente: nunca se convierten en PASSED.
Estas pruebas no acreditan integración Meta. Los asserts semánticos buscan nombres
controlados; no exigen una frase literal. Si el modelo no conserva el dato, fallan.

In [ ]:
# [CELDA 70] Campos editables para pruebas locales
# Campos editables: números sintéticos; nunca usarlos para envíos reales.
TEST_PHONE = "+15550000001"
TEST_MESSAGE = "Hola"

# Ejecución manual opcional, después de preparar Stack:
# result = process_message(TEST_PHONE, TEST_MESSAGE)
# print(result.response_text)

In [ ]:
# [CELDA 71] Pruebas de normalize_phone()
# Pruebas de normalización sin Colab, Meta, Ollama ni Llama Stack.
assert normalize_phone(TEST_PHONE) == "+15550000001"
assert normalize_phone(" +1 (555) 000-0001 ") == normalize_phone("15550000001")
assert normalize_phone("0015550000001") == "+15550000001"
assert normalize_phone("+44 7700 900123") == "+447700900123"  # Número de ficción británico.
assert normalize_phone(normalize_phone(TEST_PHONE)) == normalize_phone(TEST_PHONE)
for invalid in ("", "   ", "+", "123", "+0123456789", "++15550000001", "abc", "+15550000001 ext 2", "1" * 16):
    try:
        normalize_phone(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("Se aceptó una entrada inválida.")
try:
    normalize_phone(15550000001)
except TypeError:
    pass
else:
    raise AssertionError("Debe exigir string.")
print("NORMALIZATION ...... OK")

In [ ]:
# [CELDA 72] Prueba 1: memoria
# PRUEBA LOCAL 1 — MEMORIA. Requiere Stack real; sin mocks.
def test_memory():
    phone = "+15550000001"
    reset_local_test_state([phone])
    previous = diagnostics.enabled
    diagnostics.enabled = True
    try:
        first = process_message(phone, "Mi mascota se llama Firulais y es un perro.")
        second = process_message("+1 (555) 000-0001", "¿Cómo se llama mi mascota?")
        results = [first, second]
        show_test_results(results)
        assert first.normalized_phone == second.normalized_phone == phone
        assert first.conversation_id and first.conversation_id == second.conversation_id
        assert all(r.route == Route.LLM for r in results)
        assert llm_request_count(results) == 2
        assert "firulais" in normalized_text(second.response_text)
        print("MEMORIA: PASSED (Stack real)")
        return results
    finally:
        diagnostics.enabled = previous

memory_results = test_memory()

In [ ]:
# [CELDA 73] Prueba 2: aislamiento entre teléfonos
# PRUEBA LOCAL 2 — AISLAMIENTO. Requiere Stack real; sin mocks.
def test_isolation():
    phone_a, phone_b = "+15550000002", "+15550000003"
    reset_local_test_state([phone_a, phone_b])
    previous = diagnostics.enabled
    diagnostics.enabled = True
    try:
        a1 = process_message(phone_a, "Mi gata se llama Zafira.")
        b1 = process_message(phone_b, "Mi perro se llama Baltasar.")
        a2 = process_message(phone_a, "¿Cómo se llama mi mascota?")
        b2 = process_message(phone_b, "¿Cómo se llama mi mascota?")
        results = [a1, b1, a2, b2]
        show_test_results(results)
        assert a1.normalized_phone == a2.normalized_phone == phone_a
        assert b1.normalized_phone == b2.normalized_phone == phone_b
        assert a1.normalized_phone != b1.normalized_phone
        assert a1.conversation_id and a1.conversation_id == a2.conversation_id
        assert b1.conversation_id and b1.conversation_id == b2.conversation_id
        assert a1.conversation_id != b1.conversation_id
        assert all(r.route == Route.LLM for r in results)
        assert llm_request_count([a1, a2]) == llm_request_count([b1, b2]) == 2
        assert "zafira" in normalized_text(a2.response_text)
        assert "baltasar" not in normalized_text(a2.response_text)
        assert "baltasar" in normalized_text(b2.response_text)
        assert "zafira" not in normalized_text(b2.response_text)
        print("AISLAMIENTO: PASSED (Stack real)")
        return results
    finally:
        diagnostics.enabled = previous

isolation_results = test_isolation()

In [ ]:
# [CELDA 74] Prueba 3: escalamiento humano
# PRUEBA LOCAL 3 — ESCALAMIENTO. Ejecutable sin cliente ni servicios.
def test_human_escalation():
    phone = "+15550000004"
    reset_local_test_state([phone])
    previous = diagnostics.enabled
    diagnostics.enabled = True
    try:
        result = process_message(phone, "Quiero hablar con una persona")
        show_test_results([result])
        assert result.route == Route.HUMAN_ESCALATION
        assert result.response_text == HUMAN_ESCALATION_RESPONSE
        assert result.conversation_id is None and result.history_count is None
        assert phone not in _conversation_ids
        assert llm_request_count([result]) == 0
        assert any(e.run_id == result.run_id and e.event == EventKind.LLM_SKIPPED
                   for e in diagnostics.snapshot())
        for message in ("quiero hablar con un humano", "quiero hablar con alguien",
                        "  COMUNÍCAME   con una persona ", "necesito atención humana"):
            assert requests_human(message)
        for message in ("Mi perro se asusta con una persona desconocida",
                        "¿El champú humano sirve para perros?",
                        "No quiero hablar con una persona"):
            assert not requests_human(message)
        print("ESCALAMIENTO: PASSED (regla real, cero generaciones)")
        return result
    finally:
        diagnostics.enabled = previous

escalation_result = test_human_escalation()

In [ ]:
# [CELDA 75] Prueba 4: urgencia veterinaria
# PRUEBA LOCAL 4 — URGENCIA. Sin cliente, servicios ni Meta.
def test_veterinary_emergency():
    phone = "+15550000005"
    reset_local_test_state([phone])
    previous = diagnostics.enabled
    diagnostics.enabled = True
    try:
        messages = ["Mi perro está convulsionando y no responde.",
                    "Mi perro no puede respirar y quiero hablar con una persona."]
        results = [process_message(phone, message) for message in messages]
        show_test_results(results)
        for result in results:
            assert result.route == Route.VETERINARY_EMERGENCY
            assert result.response_text == VETERINARY_EMERGENCY_RESPONSE
            assert result.conversation_id is None and result.history_count is None
            assert result.history_trimmed is None
            assert llm_request_count([result]) == 0
            events = [e for e in diagnostics.snapshot() if e.run_id == result.run_id]
            assert any(e.event == EventKind.LLM_SKIPPED for e in events)
            assert not any(e.event == EventKind.LOCK_WAIT for e in events)
        assert phone not in _conversation_ids
        for message in ("Mi perro necesita una consulta general.",
                        "Quiero saber cuánto cuesta una revisión.",
                        "¿Cuánto cuesta un análisis de sangre?", "dolor", "enfermo", "vómito"):
            assert not is_veterinary_emergency(message)
        for message in ("NO RESPIRA", "dificultad para respirar", "convulsión", "convulsiones",
                        "sangrado abundante", "hemorragia", "atropellada", "atropellaron",
                        "envenenamiento", "envenenado", "intoxicación grave", "inconsciente",
                        "perdió el conocimiento"):
            assert is_veterinary_emergency(message)
        assert select_route("Quiero hablar con una persona") == Route.HUMAN_ESCALATION
        assert select_route("Consulta general") == Route.LLM
        print("URGENCIA: PASSED (regla real; prioridad; cero generaciones)")
        return results
    finally:
        diagnostics.enabled = previous

emergency_results = test_veterinary_emergency()

In [ ]:
# [CELDA 76] Prueba 5: recorte real de historial
# PRUEBA LOCAL 5 — RECORTE. Stack real; no fixtures ni mocks.
def test_history_trimming():
    phone = "+15550000006"
    reset_local_test_state([phone])
    previous = diagnostics.enabled
    diagnostics.enabled = True
    try:
        results = []
        # Con mensajes textuales normales son al menos ocho items tras cuatro turnos.
        for turn in range(4):
            results.append(process_message(phone,
                f"Turno de prueba {turn + 1}. Mi mascota se llama Nacarina. Confirma su nombre brevemente."))
        assert all(r.route == Route.LLM and r.history_count <= MAX_HISTORY_ITEMS for r in results)
        assert any(r.history_trimmed for r in results), "No se provocó recorte real."
        initial_id, active_before_followup = results[0].conversation_id, results[-1].conversation_id
        phone_lock, _ = get_phone_lock(phone)
        with phone_lock:
            assert _conversation_ids[phone] == active_before_followup
            assert len(read_history_items(active_before_followup)) <= MAX_HISTORY_ITEMS
        followup = process_message(phone, "¿Cómo se llama mi mascota?")
        results.append(followup)
        assert followup.route == Route.LLM and followup.response_text.strip()
        assert "nacarina" in normalized_text(followup.response_text)
        assert llm_request_count(results) == 5
        run_ids = {r.run_id for r in results}
        events = [e for e in diagnostics.snapshot() if e.run_id in run_ids]
        trims = [e for e in events if e.event == EventKind.HISTORY_TRIM_COMPLETED]
        rotations = [e for e in events if e.event == EventKind.CONVERSATION_ROTATED]
        assert trims and rotations
        assert not any(e.event == EventKind.HISTORY_TRIM_FAILED for e in events)
        for event in trims:
            assert event.history_before > MAX_HISTORY_ITEMS
            assert event.history_after == MAX_HISTORY_ITEMS
            assert any(e.run_id == event.run_id and e.event == EventKind.HISTORY_TRIM_STARTED
                       and e.sequence < event.sequence for e in events)
        # La generación posterior empezó con el ID activo tras el primer recorte.
        assert any(e.run_id == followup.run_id and e.event == EventKind.LLM_REQUEST_STARTED
                   and e.conversation_id == active_before_followup for e in events)
        with phone_lock:
            assert _conversation_ids[phone] == followup.conversation_id
            final_items = read_history_items(followup.conversation_id)
        assert len(final_items) == followup.history_count <= MAX_HISTORY_ITEMS
        assert initial_id != followup.conversation_id
        show_test_results(results)
        print({"initial_conversation_id": initial_id, "final_conversation_id": followup.conversation_id,
               "history_count_before": trims[0].history_before, "history_count_final": len(final_items)})
        for event in events:
            if event.event in (EventKind.HISTORY_TRIM_STARTED, EventKind.HISTORY_TRIM_COMPLETED,
                               EventKind.HISTORY_TRIM_FAILED, EventKind.CONVERSATION_ROTATED):
                print({"event": event.event.value, "run_id": event.run_id, "sequence": event.sequence,
                       "old_id": event.old_conversation_id, "new_id": event.new_conversation_id,
                       "before": event.history_before, "after": event.history_after})
        print("RECORTE: PASSED (borrado/rotación nativos y continuidad real)")
        return results
    finally:
        diagnostics.enabled = previous

trimming_results = test_history_trimming()

In [ ]:
# [CELDA 77] Prueba 6: concurrencia
# PRUEBA LOCAL 6 — CONCURRENCIA. Threads y Stack real; sin mocks.
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Barrier


def concurrency_evidence(phones, *, after_sequence=0):
    # Solo teléfonos sintéticos. No imprimir prompts, respuestas, excepciones completas
    # ni configuración HTTP. Duración de request incluye cola/red/modelo/extracción.
    assert set(phones) <= SYNTHETIC_PHONES
    events = [e for e in diagnostics.snapshot()
              if e.normalized_phone in phones and e.sequence > after_sequence]
    relevant = {EventKind.LOCK_WAIT, EventKind.LOCK_ACQUIRED, EventKind.LOCK_RELEASED,
                EventKind.LLM_REQUEST_STARTED, EventKind.LLM_REQUEST_COMPLETED,
                EventKind.LLM_REQUEST_FAILED, EventKind.RUN_FAILED}
    for event in events:
        if event.event in relevant:
            print({"event": event.event.value, "phone": event.normalized_phone,
                   "run_id": event.run_id, "lock_id": event.lock_id,
                   "sequence": event.sequence, "timestamp_ns": event.timestamp_ns})
    for run_id in dict.fromkeys(e.run_id for e in events):
        own = [e for e in events if e.run_id == run_id]
        starts = [e for e in own if e.event == EventKind.LLM_REQUEST_STARTED]
        ends = [e for e in own if e.event in
                (EventKind.LLM_REQUEST_COMPLETED, EventKind.LLM_REQUEST_FAILED)]
        if starts:
            print({"run_id": run_id, "phone": starts[0].normalized_phone,
                   "llm_request_seconds": ((ends[-1].timestamp_ns - starts[0].timestamp_ns) / 1e9
                                           if ends else None),
                   "llm_terminal_event": ends[-1].event.value if ends else "PENDING"})
    return events


def exception_type_chain(error):
    names, seen = [], set()
    while error is not None and id(error) not in seen:
        seen.add(id(error))
        names.append(type(error).__name__)
        error = error.__cause__
    return names


def run_simultaneously(phones):
    assert set(phones) <= SYNTHETIC_PHONES
    barrier = Barrier(len(phones))
    before = diagnostics.snapshot()
    after_sequence = before[-1].sequence if before else 0

    def worker(index, phone):
        started = time.monotonic()
        result, error, stage = None, None, "barrier"
        try:
            barrier.wait(timeout=15)
            stage = "process_message"
            result = process_message(phone, "¿Qué servicios generales ofrece una veterinaria? Responde brevemente.")
        except Exception as caught:
            error = caught  # Conservar para propagar; nunca convertir en CoreResult exitoso.
        return {"worker": index, "phone": phone, "stage": stage, "result": result,
                "error": error, "elapsed_seconds": time.monotonic() - started}

    outcomes = []
    # No cerrar/mutar el cliente compartido ni resetear estado mientras hay workers.
    with ThreadPoolExecutor(max_workers=len(phones)) as pool:
        futures = [pool.submit(worker, index, phone) for index, phone in enumerate(phones)]
        for future in as_completed(futures):
            outcome = future.result()
            outcomes.append(outcome)
            print({"worker": outcome["worker"], "phone": outcome["phone"],
                   "stage": outcome["stage"], "elapsed_seconds": outcome["elapsed_seconds"],
                   "status": "FAILED" if outcome["error"] is not None else "COMPLETED",
                   "exception_types": exception_type_chain(outcome["error"])})
    concurrency_evidence(phones, after_sequence=after_sequence)
    failures = [outcome for outcome in outcomes if outcome["error"] is not None]
    if failures:
        failed = failures[0]
        raise RuntimeError(
            f"CONCURRENCIA: FAILED; worker={failed['worker']}; phone={failed['phone']}; "
            f"exception_types={exception_type_chain(failed['error'])}") from failed["error"]
    return [outcome["result"] for outcome in sorted(outcomes, key=lambda outcome: outcome["worker"])]


def lock_intervals(results):
    events = diagnostics.snapshot()
    intervals = []
    for result in results:
        own = [e for e in events if e.run_id == result.run_id]
        def one(kind):
            selected = [e for e in own if e.event == kind]
            assert len(selected) == 1, (result.run_id, kind)
            return selected[0]
        waiting, acquired, released = (one(kind) for kind in
            (EventKind.LOCK_WAIT, EventKind.LOCK_ACQUIRED, EventKind.LOCK_RELEASED))
        assert waiting.lock_id == acquired.lock_id == released.lock_id
        assert waiting.sequence < acquired.sequence < released.sequence
        assert waiting.timestamp_ns <= acquired.timestamp_ns <= released.timestamp_ns
        llm_started, llm_completed = one(EventKind.LLM_REQUEST_STARTED), one(EventKind.LLM_REQUEST_COMPLETED)
        assert acquired.sequence < llm_started.sequence < llm_completed.sequence < released.sequence
        intervals.append({"run_id": result.run_id, "lock_id": acquired.lock_id,
                          "wait": waiting.sequence, "acquired": acquired.sequence,
                          "released": released.sequence, "llm_started": llm_started.sequence,
                          "llm_completed": llm_completed.sequence})
    return intervals


def test_concurrency():
    same, phone_a, phone_b = "+15550000007", "+15550000008", "+15550000009"
    reset_local_test_state([same, phone_a, phone_b])
    previous = diagnostics.enabled
    diagnostics.enabled = True
    try:
        print("LLM_HTTP_TIMEOUT:", httpx.Timeout(120.0, read=LLM_READ_TIMEOUT_SECONDS).as_dict())
        print("LLM_MAX_RETRIES:", client.max_retries)
        assert client.max_retries == 0, "Esta prueba exige cero reintentos automáticos."
        same_results = run_simultaneously([same, same])
        same_intervals = sorted(lock_intervals(same_results), key=lambda item: item["acquired"])
        assert same_intervals[0]["lock_id"] == same_intervals[1]["lock_id"]
        assert same_intervals[0]["released"] < same_intervals[1]["acquired"]
        assert same_results[0].conversation_id == same_results[1].conversation_id
        print("SAME_PHONE: PASSED (mismo lock; exclusión mutua; misma conversación)")
        different_results = run_simultaneously([phone_a, phone_b])
        different_intervals = lock_intervals(different_results)
        assert different_intervals[0]["lock_id"] != different_intervals[1]["lock_id"]
        assert different_results[0].conversation_id != different_results[1].conversation_id
        # Además, comprobar la independencia del mecanismo sin depender del proveedor:
        # poseer A no impide adquirir B. No modificar conversaciones en este chequeo.
        lock_a, _ = get_phone_lock(phone_a)
        lock_b, _ = get_phone_lock(phone_b)
        with lock_a:
            acquired_b = lock_b.acquire(blocking=False)
            assert acquired_b
            lock_b.release()
        results = same_results + different_results
        assert all(r.route == Route.LLM and r.response_text.strip() for r in results)
        assert llm_request_count(same_results) == llm_request_count(different_results) == 2
        show_test_results(results)
        print({"same_phone_intervals": same_intervals, "different_phone_intervals": different_intervals})
        # Evidencia obligatoria en nuestra capa; no significa paralelismo físico en Ollama.
        a, b = different_intervals
        lock_overlap = max(a["acquired"], b["acquired"]) < min(a["released"], b["released"])
        llm_overlap = max(a["llm_started"], b["llm_started"]) < min(a["llm_completed"], b["llm_completed"])
        print("DIFFERENT_PHONE_LOCK_OVERLAP:", lock_overlap)
        print("DIFFERENT_PHONE_LLM_REQUEST_OVERLAP:", llm_overlap)
        assert lock_overlap, "No se observó concurrencia de procesamiento entre teléfonos."
        assert llm_overlap, "No se observaron solicitudes LLM simultáneamente en vuelo."
        print("CONCURRENCIA: PASSED (4 generaciones reales; mismo teléfono exclusivo; teléfonos distintos concurrentes)")
        return results
    except Exception as error:
        print("CONCURRENCIA: FAILED; exception_types:", exception_type_chain(error))
        concurrency_evidence([same, phone_a, phone_b])
        raise
    finally:
        diagnostics.enabled = previous

concurrency_results = test_concurrency()